# SolveBench -- Reordering Probe

The same experiment as the full reordering study, on 200 matrices sampled across the corpus's size deciles and without the exact spectra.

It exists to show the signal early: the full sweep runs for hours because the method working is what makes it expensive -- 491 systems that short-circuit as undefined suddenly have a usable diagonal and iterate to the cap.

In [ ]:
# Pin BLAS threads BEFORE numpy is imported, or the setting is ignored.
# Runtime on a shared machine is otherwise unreproducible: numpy's dense
# operations spawn a thread count that depends on the host and its load, and
# that variation lands directly in the timing column.
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"
print("BLAS threads pinned to 1 for reproducible timing")

In [ ]:
# The solvebench package, embedded verbatim from src/solvebench/ by
# tools/build_notebooks.py. Do not edit here: edit the library and
# regenerate, or the notebook and the library drift apart again.
import sys, pathlib, json

_SOURCES = json.loads(r"""{"config": "\"\"\"Every tunable number in the benchmark, in one place.\n\nThe notebook used to carry its own copy of these values, which is how the\nreleased library ended up capping direct methods at n=3000 while the run that\nproduced the results capped them at n=2000. Nothing outside this module is\nallowed to hard-code a limit or a tolerance.\n\"\"\"\n\n# --- iterative solvers -------------------------------------------------------\nMAX_ITERATIONS = 10_000     # cap before a stationary/Krylov method is called non-convergent\nTOLERANCE = 1e-8            # relative residual an iterative method is asked to reach\nSOR_OMEGA = 1.25            # fixed relaxation factor (1.0 would reduce SOR to Gauss-Seidel)\n\n# Power iterations spent estimating rho(T_J) before choosing omega adaptively.\n# 60 is enough for the two-digit accuracy the omega formula needs, and it is\n# charged to the method as 60 matvecs -- about 0.6% of the iteration cap.\nPOWER_ITERS_OMEGA = 60\n\n# A run whose residual climbs past this multiple of its own best is abandoned early:\n# it is diverging, and letting it burn the full iteration cap teaches us nothing.\n# 2,022,191 of the iterations in the first sweep were spent this way.\n#\n# Raised from 1e4 after it was caught aborting genuine convergences. rho(T) governs the\n# *asymptotic* rate; when the iteration matrix is non-normal, ||T^k|| can grow a long way\n# before it decays. cdde6 has rho(T_J) = 0.717 and ||T_J||_2 = 1.244: its residual climbs\n# to 44,764x the starting value by iteration 73 and then converges at iteration 178. The\n# old threshold killed it at 61, on the way up the hump.\n#\n# Measured cost of the old value: about 23 legitimate convergences lost across the three\n# stationary methods, 17 of them SOR -- over-relaxation amplifies exactly this transient.\n# Measured cost of the new one: genuinely divergent runs are caught at a median of 2\n# iterations and grow by orders of magnitude per step, so they still abort within a few\n# extra iterations.\nDIVERGENCE_GROWTH = 1e12\n\n# --- iterative refinement ----------------------------------------------------\n# Refinement is an ORTHOGONAL factor, not a property of one solver. The previous\n# run applied it only to APK, which is why APK appeared to be the most accurate\n# method: the same wrapper around Gauss-Seidel reaches 5.44e-16, about 27x more\n# accurate than refined APK's 1.45e-14. Every solver is now measured at each of\n# these pass counts so the comparison is like-for-like.\nREFINEMENT_PASSES = (0, 1)\n# A second pass costs roughly another full solve and buys little: measured on\n# fs_541_2, Gauss-Seidel goes 7.12e-04 -> 1.30e-11 -> 5.85e-12 across 0, 1 and 2\n# passes. Stopping at one is therefore a claim that needs evidence, so the\n# ablation below runs all three on a stratified subsample to justify it.\nREFINEMENT_STUDY_PASSES = (0, 1, 2)\n\n# --- ILU preconditioner ------------------------------------------------------\nILU_DROP_TOL = 1e-3\nILU_FILL_FACTOR = 5\n\n# --- direct solvers ----------------------------------------------------------\n# The direct solvers are hand-written rather than LAPACK calls, so they cost\n# O(n^3) in Python: roughly 50-100x slower than numpy.linalg.solve, measured at\n# ~35s for n=3000. They are pedagogical implementations validated against LAPACK,\n# not performance competitors, and are capped so a sweep stays tractable.\nDIRECT_SIZE_CAP = 2000\n\n# --- condition number --------------------------------------------------------\nCOND_EXACT_CAP = 2000       # above this, fall back to a 1-norm estimate\n# Beyond this the matrix is numerically singular in double precision and any\n# forward error reported for it says more about the matrix than about the solver.\n# 160 of 930 matrices in this collection are past it, and they are reported as a\n# separate stratum rather than pooled into the headline error statistics.\nILL_CONDITIONED = 1e15\n\n# --- spectral analysis -------------------------------------------------------\n# Above this size we estimate rho(T) by power iteration instead of forming the\n# iteration matrix; 611 of 930 matrices are small enough for the exact route.\nSPECTRAL_EXACT_CAP = 2000\nPOWER_ITER_MAX = 200\nPOWER_ITER_TOL = 1e-6\n", "metrics": "\"\"\"The single place where a solve is judged correct or not.\n\nEvery method -- direct or iterative, hand-written or library -- is scored here\nand nowhere else. This matters more than it looks. The earlier harness used two\ndifferent definitions of success: direct methods were marked \"ok\" whenever the\ncall did not raise, and iterative methods were marked \"ok\" whenever the solver\nreturned its own convergence flag. Neither one looked at the answer. That let\nGauss-Jordan report success on 57 systems whose relative residual reached 9.7e12.\n\nThe rule here is deliberately blunt: a solve succeeded if the residual we\nmeasure ourselves, afterwards, from A, x and b, is small. A solver's opinion of\nits own convergence is recorded (see `reported_converged`) but never decides\nthe outcome.\n\"\"\"\nimport numpy as np\n\n# Relative residual ||Ax - b|| / ||b|| below which a solve counts as successful.\n# Matches the tolerance the iterative solvers are asked to reach, so a converged\n# iterative solve and a good direct solve are held to the same standard.\nSUCCESS_TOL = 1e-8\n\n# Relative forward error ||x - x_true|| / ||x_true|| reported alongside, but NOT\n# used to gate success: on ill-conditioned systems a perfectly good solver can\n# have a large forward error through no fault of its own. It is a separate\n# column so both views can be reported, never a hidden second criterion.\nACCURATE_TOL = 1e-6\n\nSTATUS_SOLVED = \"solved\"                    # residual verified below SUCCESS_TOL\nSTATUS_INACCURATE = \"inaccurate\"            # solver CLAIMED success; the residual says no\nSTATUS_NO_CONVERGE = \"did_not_converge\"     # solver admitted failure, residual agrees\nSTATUS_DIVERGED = \"diverged\"                # iterate blew up or went non-finite\nSTATUS_NOT_APPLICABLE = \"not_applicable\"    # method undefined on this matrix\nSTATUS_SINGULAR = \"structurally_singular\"   # matrix has no unique solution\nSTATUS_SKIPPED = \"skipped_too_large\"        # deliberately not attempted at this size\nSTATUS_ERROR = \"error\"                      # unexpected failure, see note\n\n#: Statuses that mean \"the method was defined on this matrix and we let it try\".\n#: The denominator for a conditional success rate is exactly this set.\nAPPLICABLE_STATUSES = frozenset({STATUS_SOLVED, STATUS_INACCURATE,\n                                 STATUS_NO_CONVERGE, STATUS_DIVERGED})\n\n\ndef score(A, x, b, x_true, b_norm=None, x_true_norm=None, reported_converged=None):\n    \"\"\"Measure one solve and decide whether it succeeded.\n\n    Returns the metric columns plus a status. `reported_converged` is whatever\n    the solver claimed about itself; it is stored for comparison but has no\n    influence on the verdict.\n    \"\"\"\n    b_norm = b_norm if b_norm else (np.linalg.norm(b) or 1.0)\n    x_true_norm = x_true_norm if x_true_norm else (np.linalg.norm(x_true) or 1.0)\n\n    if x is None or not np.all(np.isfinite(x)):\n        return {\"status\": STATUS_DIVERGED, \"residual_abs\": np.nan, \"error_abs\": np.nan,\n                \"residual_rel\": np.nan, \"error_rel\": np.nan,\n                \"reported_converged\": reported_converged, \"accurate\": False}\n\n    residual_abs = float(np.linalg.norm(A @ x - b))\n    error_abs = float(np.linalg.norm(x - x_true))\n    residual_rel = residual_abs / b_norm\n    error_rel = error_abs / x_true_norm\n\n    if residual_rel <= SUCCESS_TOL:\n        status = STATUS_SOLVED\n    elif reported_converged is False:\n        status = STATUS_NO_CONVERGE     # the solver said so itself, and it was right\n    else:\n        # The solver either claimed convergence or, like the hand-written direct\n        # methods, simply returned without complaint -- and the answer is wrong.\n        # This is the category that was invisible before: 57 Gauss-Jordan solves\n        # were counted as successes here, one of them at a relative residual of\n        # 9.71e+12.\n        status = STATUS_INACCURATE\n\n    return {\n        \"status\": status,\n        \"residual_abs\": residual_abs,\n        \"error_abs\": error_abs,\n        \"residual_rel\": residual_rel,\n        \"error_rel\": error_rel,\n        \"reported_converged\": reported_converged,\n        \"accurate\": bool(error_rel <= ACCURATE_TOL),\n    }\n\n\ndef blank(status, note=None):\n    \"\"\"A metric row for a solve that never happened (skipped, singular, N/A).\"\"\"\n    row = {\"status\": status, \"residual_abs\": np.nan, \"error_abs\": np.nan,\n           \"residual_rel\": np.nan, \"error_rel\": np.nan,\n           \"reported_converged\": None, \"accurate\": False}\n    if note is not None:\n        row[\"note\"] = note\n    return row\n\n\ndef rates(df, method):\n    \"\"\"Applicability and conditional success for one method, as separate numbers.\n\n    These must never be multiplied together into a single \"success rate\": doing\n    that is what produced the earlier 10.4% figure for Conjugate Gradient, whose\n    real conditional success is 83.9%. Report both, always with the denominator.\n    \"\"\"\n    rows = df[df[\"method\"] == method]\n    total = len(rows)\n    applicable = rows[\"status\"].isin(APPLICABLE_STATUSES).sum()\n    solved = (rows[\"status\"] == STATUS_SOLVED).sum()\n    return {\n        \"method\": method,\n        \"corpus\": total,\n        \"applicable\": int(applicable),\n        \"solved\": int(solved),\n        \"applicability\": applicable / total if total else float(\"nan\"),\n        \"conditional_success\": solved / applicable if applicable else float(\"nan\"),\n    }\n", "direct_solvers": "\"\"\"Four direct solvers for Ax = b, implemented from scratch on dense arrays\nwith vectorized NumPy row operations (not element-by-element Python loops --\nsee the guide's runtime section for why that distinction matters at this scale).\n\nEach solver raises SolverNotApplicable if the matrix doesn't meet its\nrequirements (e.g. Cholesky needs symmetric positive-definite).\n\"\"\"\nimport numpy as np\n\n\nclass SolverNotApplicable(Exception):\n    \"\"\"Raised when a method's mathematical requirements aren't met by this matrix.\"\"\"\n\n\ndef _partial_pivot(A: np.ndarray, b: np.ndarray, row: int) -> None:\n    \"\"\"In-place: swap row `row` with the row below it that has the largest\n    pivot-column magnitude, to keep elimination numerically stable.\n    \"\"\"\n    n = A.shape[0]\n    pivot_row = row + int(np.argmax(np.abs(A[row:, row])))\n    if abs(A[pivot_row, row]) < 1e-14:\n        raise SolverNotApplicable(\"matrix is singular (zero pivot even after partial pivoting)\")\n    if pivot_row != row:\n        A[[row, pivot_row]] = A[[pivot_row, row]]\n        b[[row, pivot_row]] = b[[pivot_row, row]]\n\n\ndef gauss_elimination(A_dense: np.ndarray, b: np.ndarray) -> tuple[np.ndarray, int]:\n    \"\"\"Forward elimination with partial pivoting, then back-substitution.\n\n    Returns (x, steps), where steps is the number of elimination stages\n    actually performed (n-1 for a full-size solve) -- a direct method's\n    deterministic analogue of \"iterations\": fixed by matrix size, not by\n    how close the current estimate is to converged.\n    \"\"\"\n    n = A_dense.shape[0]\n    A = A_dense.copy()\n    b = b.copy()\n    steps = 0\n\n    for k in range(n - 1):\n        _partial_pivot(A, b, k)\n        factors = A[k + 1 :, k] / A[k, k]\n        A[k + 1 :, k:] -= np.outer(factors, A[k, k:])\n        b[k + 1 :] -= factors * b[k]\n        steps += 1\n\n    x = np.zeros(n)\n    for i in range(n - 1, -1, -1):\n        x[i] = (b[i] - A[i, i + 1 :] @ x[i + 1 :]) / A[i, i]\n    return x, steps\n\n\ndef gauss_jordan(A_dense: np.ndarray, b: np.ndarray) -> tuple[np.ndarray, int]:\n    \"\"\"Reduce all the way to the identity matrix (no separate back-substitution).\n\n    Returns (x, steps): one elimination stage per row, so steps = n.\n    \"\"\"\n    n = A_dense.shape[0]\n    A = A_dense.copy()\n    b = b.copy()\n    steps = 0\n\n    for k in range(n):\n        _partial_pivot(A, b, k)\n        pivot = A[k, k]\n        A[k, :] /= pivot\n        b[k] /= pivot\n        pivot_col = A[:, k].copy()\n        pivot_col[k] = 0.0\n        A -= np.outer(pivot_col, A[k, :])\n        b -= pivot_col * b[k]\n        steps += 1\n\n    return b, steps\n\n\ndef lu_solve(A_dense: np.ndarray, b: np.ndarray) -> tuple[np.ndarray, int]:\n    \"\"\"Doolittle LU decomposition with partial pivoting (A = P^-1 L U), then\n    solve via forward substitution (Ly=b) followed by back substitution (Ux=y).\n\n    Returns (x, steps): steps = n-1 factorization stages (same structure as\n    Gauss elimination, since LU is elimination with the multipliers kept).\n    \"\"\"\n    n = A_dense.shape[0]\n    U = A_dense.copy()\n    L = np.eye(n)\n    perm_b = b.copy()\n    steps = 0\n\n    for k in range(n - 1):\n        pivot_row = k + int(np.argmax(np.abs(U[k:, k])))\n        if abs(U[pivot_row, k]) < 1e-14:\n            raise SolverNotApplicable(\"matrix is singular (zero pivot even after partial pivoting)\")\n        if pivot_row != k:\n            U[[k, pivot_row]] = U[[pivot_row, k]]\n            perm_b[[k, pivot_row]] = perm_b[[pivot_row, k]]\n            if k > 0:\n                L[[k, pivot_row], :k] = L[[pivot_row, k], :k]\n        factors = U[k + 1 :, k] / U[k, k]\n        L[k + 1 :, k] = factors\n        U[k + 1 :, k:] -= np.outer(factors, U[k, k:])\n        steps += 1\n\n    # Forward substitution: L y = perm_b\n    y = np.zeros(n)\n    for i in range(n):\n        y[i] = perm_b[i] - L[i, :i] @ y[:i]\n\n    # Back substitution: U x = y\n    x = np.zeros(n)\n    for i in range(n - 1, -1, -1):\n        x[i] = (y[i] - U[i, i + 1 :] @ x[i + 1 :]) / U[i, i]\n    return x, steps\n\n\ndef cholesky_solve(A_dense: np.ndarray, b: np.ndarray, sym_tol: float = 1e-8) -> tuple[np.ndarray, int]:\n    \"\"\"Cholesky decomposition (A = R^T R), valid only for symmetric\n    positive-definite matrices. Raises SolverNotApplicable otherwise.\n\n    Returns (x, steps): one factorization stage per row, so steps = n.\n    \"\"\"\n    n = A_dense.shape[0]\n    if not np.allclose(A_dense, A_dense.T, atol=sym_tol, rtol=sym_tol):\n        raise SolverNotApplicable(\"matrix is not symmetric\")\n\n    R = np.zeros((n, n))\n    steps = 0\n    for i in range(n):\n        diag_term = A_dense[i, i] - R[:i, i] @ R[:i, i]\n        if diag_term <= 0:\n            raise SolverNotApplicable(\"matrix is not positive-definite\")\n        R[i, i] = np.sqrt(diag_term)\n        if i + 1 < n:\n            R[i, i + 1 :] = (A_dense[i, i + 1 :] - R[:i, i] @ R[:i, i + 1 :]) / R[i, i]\n        steps += 1\n\n    # Solve R^T y = b (forward), then R x = y (back)\n    y = np.zeros(n)\n    for i in range(n):\n        y[i] = b[i] - R[:i, i] @ y[:i]\n        y[i] /= R[i, i]\n\n    x = np.zeros(n)\n    for i in range(n - 1, -1, -1):\n        x[i] = (y[i] - R[i, i + 1 :] @ x[i + 1 :]) / R[i, i]\n    return x, steps\n", "iterative_solvers": "\"\"\"Iterative solvers for Ax = b on sparse matrices.\n\nEvery solver returns ``(x, iterations, reported_converged, work)``. The third\nvalue is the solver's own opinion and is recorded but never trusted -- success\nis decided afterwards by :mod:`solvebench.metrics` from the residual we measure\nourselves. The fourth is a :class:`Work` tally of the operations that actually\ncost something, which is the cost metric the comparison uses: wall-clock on a\nshared Kaggle CPU is not reproducible, and \"iterations\" are not comparable\nacross methods because one preconditioned step does far more work than one\nJacobi step.\n\nTwo implementation notes matter for the fairness of the comparison:\n\n* Gauss-Seidel and SOR are written in *delta form*, ``(D + wL) d = w r`` with\n  ``x <- x + d``. The previous version solved ``(D+L) x = b - Ux`` by calling\n  ``spsolve_triangular`` inside the loop, which re-analysed the triangular\n  structure on every one of up to 10,000 iterations, and then computed the\n  residual with a second matrix-vector product. Delta form factors the\n  triangular matrix once and gets the residual for free, so what the timer sees\n  is the algorithm rather than repeated setup.\n* Jacobi is written the same way for the same reason: ``x <- x + r/d``.\n\"\"\"\nimport numpy as np\nimport scipy.sparse as sp\nimport scipy.sparse.linalg as spla\n\nfrom . import config\nfrom .direct_solvers import SolverNotApplicable\n\n\nclass Work:\n    \"\"\"Counts the operations a solve actually performed.\n\n    Wall-clock timing on shared hardware is not reproducible and iteration\n    counts are not comparable between methods, so this is the primary cost\n    metric. A preconditioner application is counted separately from a plain\n    matrix-vector product because it is typically far more expensive.\n    \"\"\"\n\n    __slots__ = (\"matvecs\", \"tri_solves\", \"precond\", \"setup\")\n\n    def __init__(self):\n        self.matvecs = 0\n        self.tri_solves = 0\n        self.precond = 0\n        self.setup = 0.0      # seconds spent building a factorization, if any\n\n    def as_dict(self):\n        return {\"matvecs\": self.matvecs, \"tri_solves\": self.tri_solves,\n                \"precond_applies\": self.precond, \"setup_sec\": self.setup}\n\n\ndef _counting_operator(A, work):\n    \"\"\"Wrap A so that every product a library solver takes gets counted.\"\"\"\n    def matvec(v):\n        work.matvecs += 1\n        return A @ v\n    return spla.LinearOperator(A.shape, matvec=matvec, dtype=np.float64)\n\n\ndef _counting_preconditioner(apply_fn, shape, work):\n    def matvec(v):\n        work.precond += 1\n        return apply_fn(v)\n    return spla.LinearOperator(shape, matvec=matvec, dtype=np.float64)\n\n\ndef _diagonal_or_raise(A, what):\n    d = A.diagonal()\n    if np.any(np.abs(d) < 1e-14):\n        raise SolverNotApplicable(f\"{what} needs a nonzero diagonal \"\n                                  f\"({int((np.abs(d) < 1e-14).sum())} zero entries)\")\n    return d\n\n\ndef _prefactor_lower(M):\n    \"\"\"Factor a lower-triangular iteration matrix once, for repeated solves.\n\n    ``permc_spec=\"NATURAL\"`` and ``diag_pivot_thresh=0`` keep SuperLU from\n    reordering or pivoting, so the factorization of an already-triangular\n    matrix is essentially free and every later solve is a plain substitution.\n    \"\"\"\n    return spla.splu(M.tocsc(), permc_spec=\"NATURAL\", diag_pivot_thresh=0.0)\n\n\n# --------------------------------------------------------------- stationary\n\n\ndef jacobi(A, b, max_iter=config.MAX_ITERATIONS, tol=config.TOLERANCE):\n    \"\"\"x <- x + D^-1 (b - Ax). One matrix-vector product per iteration, and the\n    residual needed for the stopping test falls out of it at no extra cost.\"\"\"\n    work = Work()\n    d = _diagonal_or_raise(A, \"Jacobi\")\n    b_norm = np.linalg.norm(b) or 1.0\n    x = np.zeros(A.shape[0])\n    best = np.inf\n\n    for k in range(1, max_iter + 1):\n        r = b - A @ x\n        work.matvecs += 1\n        rr = np.linalg.norm(r) / b_norm\n        if not np.isfinite(rr):\n            return x, k, False, work\n        if rr <= tol:\n            return x, k, True, work\n        if rr > best * config.DIVERGENCE_GROWTH:\n            return x, k, False, work\n        best = min(best, rr)\n        x = x + r / d\n\n    return x, max_iter, False, work\n\n\ndef _sor_core(A, b, omega, max_iter, tol, label):\n    \"\"\"Shared engine for Gauss-Seidel (omega = 1) and SOR.\n\n    Delta form: (D + omega*L) delta = omega * r, then x <- x + delta. Writing\n    both methods through one routine guarantees they are treated identically;\n    the only difference between them is omega.\n    \"\"\"\n    work = Work()\n    _diagonal_or_raise(A, label)\n    n = A.shape[0]\n    b_norm = np.linalg.norm(b) or 1.0\n\n    M = (sp.tril(A, k=-1, format=\"csr\") * omega + sp.diags(A.diagonal())).tocsc()\n    try:\n        lu = _prefactor_lower(M)\n    except RuntimeError as e:                      # singular triangular part\n        raise SolverNotApplicable(f\"{label}: {e}\") from e\n\n    x = np.zeros(n)\n    best = np.inf\n\n    for k in range(1, max_iter + 1):\n        r = b - A @ x\n        work.matvecs += 1\n        rr = np.linalg.norm(r) / b_norm\n        if not np.isfinite(rr):\n            return x, k, False, work\n        if rr <= tol:\n            return x, k, True, work\n        if rr > best * config.DIVERGENCE_GROWTH:\n            return x, k, False, work\n        best = min(best, rr)\n        x = x + lu.solve(omega * r)\n        work.tri_solves += 1\n\n    return x, max_iter, False, work\n\n\ndef gauss_seidel(A, b, max_iter=config.MAX_ITERATIONS, tol=config.TOLERANCE):\n    return _sor_core(A, b, 1.0, max_iter, tol, \"Gauss-Seidel\")\n\n\ndef sor(A, b, omega=config.SOR_OMEGA, max_iter=config.MAX_ITERATIONS, tol=config.TOLERANCE):\n    return _sor_core(A, b, omega, max_iter, tol, \"SOR\")\n\ndef estimate_rho_jacobi(A, iters=None, seed=0, work=None):\n    \"\"\"Power iteration on ``T_J = I - D^-1 A``, returning an estimate of rho(T_J).\n\n    Uses only matrix-vector products, so it costs the same as ``iters`` Jacobi steps --\n    and those matvecs are added to ``work``. Charging for the estimate matters: an\n    adaptive method that hides its own setup would beat a fixed one on cost by\n    bookkeeping rather than by being cheaper.\n\n    Returns NaN when the diagonal carries a zero, since T_J does not exist there.\n    \"\"\"\n    iters = config.POWER_ITERS_OMEGA if iters is None else iters\n    d = A.diagonal()\n    if np.any(np.abs(d) < 1e-14):\n        return float(\"nan\")\n    rng = np.random.default_rng(seed)\n    v = rng.normal(size=A.shape[0])\n    nv = np.linalg.norm(v)\n    if nv == 0:\n        return float(\"nan\")\n    v /= nv\n    lam = 0.0\n    for _ in range(iters):\n        w = v - (A @ v) / d\n        if work is not None:\n            work.matvecs += 1\n        nw = np.linalg.norm(w)\n        if nw < 1e-300:\n            return 0.0\n        lam = nw\n        v = w / nw\n    return float(lam)\n\n\ndef optimal_omega(rho):\n    \"\"\"Young (1950): ``omega* = 2 / (1 + sqrt(1 - rho(T_J)^2))``.\n\n    The formula is exact for a consistently ordered matrix with property A. Most of this\n    corpus is neither, so here it is a heuristic and is reported as one -- what the\n    benchmark measures is whether it is worth applying outside its hypotheses.\n\n    Falls back to 1.0 -- plain Gauss-Seidel -- when no usable estimate exists. That is\n    the safe direction: rho >= 1 means the iteration is not converging anyway, and\n    over-relaxing a divergent iteration makes it worse faster.\n    \"\"\"\n    if not np.isfinite(rho) or rho >= 1.0:\n        return 1.0\n    return float(np.clip(2.0 / (1.0 + np.sqrt(max(1.0 - rho * rho, 0.0))), 1.0, 1.95))\n\n\ndef sor_adaptive(A, b, max_iter=config.MAX_ITERATIONS, tol=config.TOLERANCE):\n    \"\"\"SOR with omega chosen per matrix instead of fixed.\n\n    A single relaxation factor for every matrix is measurably the wrong call in both\n    directions: on this corpus plain Gauss-Seidel solves 11 systems that SOR at 1.25\n    misses, while SOR at 1.25 solves 9 that Gauss-Seidel misses. Choosing omega from an\n    estimate of rho(T_J) recovers both sides.\n    \"\"\"\n    work = Work()\n    rho = estimate_rho_jacobi(A, work=work)\n    omega = optimal_omega(rho)\n    x, iters, converged, w2 = _sor_core(A, b, omega, max_iter, tol, \"SOR (adaptive)\")\n    w2.matvecs += work.matvecs          # the estimate is part of this method's cost\n    return x, iters, converged, w2\n\n# --------------------------------------------------------------- Krylov\n\n\ndef _is_symmetric(A, tol=1e-8):\n    diff = abs(A - A.T)\n    return diff.nnz == 0 or diff.max() <= tol * max(abs(A).max(), 1.0)\n\n\ndef conjugate_gradient(A, b, max_iter=config.MAX_ITERATIONS, tol=config.TOLERANCE):\n    \"\"\"Unpreconditioned CG. Requires symmetry; indefiniteness shows up as a\n    non-positive curvature term and is reported rather than silently ignored.\"\"\"\n    if not _is_symmetric(A):\n        raise SolverNotApplicable(\"Conjugate Gradient needs a symmetric matrix\")\n    work = Work()\n    b_norm = np.linalg.norm(b) or 1.0\n    x = np.zeros(A.shape[0])\n    r = b.copy()\n    p = r.copy()\n    rs = r @ r\n\n    # x = 0 already solves it. Without this the first search direction is the\n    # zero vector, p'Ap is 0, and the matrix gets reported as indefinite.\n    if np.sqrt(rs) <= tol * b_norm:\n        return x, 0, True, work\n\n    for k in range(1, max_iter + 1):\n        Ap = A @ p\n        work.matvecs += 1\n        pAp = p @ Ap\n        if pAp <= 0:\n            raise SolverNotApplicable(\"Conjugate Gradient needs positive definiteness \"\n                                      f\"(p'Ap = {pAp:.3e} at iteration {k})\")\n        alpha = rs / pAp\n        x = x + alpha * p\n        r = r - alpha * Ap\n        rr = np.linalg.norm(r) / b_norm\n        if not np.isfinite(rr):\n            return x, k, False, work\n        if rr <= tol:\n            return x, k, True, work\n        rs_new = r @ r\n        p = r + (rs_new / rs) * p\n        rs = rs_new\n\n    return x, max_iter, False, work\n\n\ndef bicgstab(A, b, M=None, max_iter=config.MAX_ITERATIONS, tol=config.TOLERANCE):\n    \"\"\"BiCGSTAB via scipy, with A and the preconditioner wrapped so their\n    applications are counted. Unlike the hand-written stationary methods this is\n    a library implementation, and the results table labels it as such.\"\"\"\n    work = Work()\n    op = _counting_operator(A, work)\n    pre = None if M is None else _counting_preconditioner(M, A.shape, work)\n    x, info = spla.bicgstab(op, b, rtol=tol, atol=0.0, maxiter=max_iter, M=pre)\n    return x, work.matvecs, info == 0, work\n\n\ndef pcg(A, b, M=None, max_iter=config.MAX_ITERATIONS, tol=config.TOLERANCE):\n    work = Work()\n    op = _counting_operator(A, work)\n    pre = None if M is None else _counting_preconditioner(M, A.shape, work)\n    x, info = spla.cg(op, b, rtol=tol, atol=0.0, maxiter=max_iter, M=pre)\n    return x, work.matvecs, info == 0, work\n\n\ndef gmres(A, b, M=None, restart=30, max_iter=config.MAX_ITERATIONS, tol=config.TOLERANCE):\n    \"\"\"Restarted GMRES -- the default Krylov method in PETSc, and the baseline\n    whose absence made the dispatched solver look better than it is.\"\"\"\n    work = Work()\n    op = _counting_operator(A, work)\n    pre = None if M is None else _counting_preconditioner(M, A.shape, work)\n    x, info = spla.gmres(op, b, rtol=tol, atol=0.0, restart=restart,\n                         maxiter=max_iter // restart or 1, M=pre)\n    return x, work.matvecs, info == 0, work\n\n\n# --------------------------------------------------------------- preconditioning\n\n\ndef build_ilu(A, work=None):\n    \"\"\"Incomplete LU with the same relax-and-retry ladder the earlier run used.\n\n    Returns an apply function, or None if no usable factorization was found.\n    Note for the write-up: SuperLU's spilu applies a column permutation and\n    partial pivoting, so the resulting factor is *not* symmetric even when A is.\n    Feeding it to CG as a preconditioner violates CG's requirement that the\n    preconditioner be symmetric positive definite, which is why the dispatched\n    solver's symmetric branch is reported as a known defect rather than a\n    feature.\n    \"\"\"\n    import time\n    t0 = time.perf_counter()\n    for drop, fill in ((config.ILU_DROP_TOL, config.ILU_FILL_FACTOR),\n                       (config.ILU_DROP_TOL * 10, config.ILU_FILL_FACTOR * 2),\n                       (1e-2, 20)):\n        try:\n            ilu = spla.spilu(A.tocsc(), drop_tol=drop, fill_factor=fill)\n        except (RuntimeError, ValueError, MemoryError):\n            continue\n        if work is not None:\n            work.setup += time.perf_counter() - t0\n        return ilu.solve\n\n    d = A.diagonal()                          # last resort: diagonal scaling\n    if np.any(np.abs(d) < 1e-14):\n        return None\n    if work is not None:\n        work.setup += time.perf_counter() - t0\n    return lambda v: v / d\n", "reference_solvers": "\"\"\"Library solvers: the ones an engineer with this problem would actually reach for.\n\nTheir absence from the first sweep is what made the proposed method look strong.\nA comparative study of solvers for sparse systems that never runs\n``scipy.sparse.linalg.spsolve`` has not compared against the state of practice,\nit has compared against four textbook methods from the 1950s.\n\nThese are sparse direct factorizations, so unlike the hand-written dense direct\nsolvers in :mod:`solvebench.direct_solvers` they carry no size cap -- SuperLU on\na sparse matrix costs far less than O(n^3), which is exactly the point being\ntested.\n\nEvery solver here returns the same ``(x, iterations, reported_converged, work)``\ntuple as the iterative ones so the harness can score them identically.\n\"\"\"\nimport time\n\nimport numpy as np\nimport scipy.sparse.linalg as spla\n\nfrom .direct_solvers import SolverNotApplicable\nfrom .iterative_solvers import Work, build_ilu, bicgstab, gmres, pcg, _is_symmetric\n\n\ndef sparse_lu(A, b):\n    \"\"\"SuperLU factorization then a single triangular solve pair.\n\n    Reported as one \"iteration\" because a direct method does a fixed amount of\n    work: there is no convergence loop to count.\n    \"\"\"\n    work = Work()\n    t0 = time.perf_counter()\n    try:\n        lu = spla.splu(A.tocsc())\n    except RuntimeError as e:\n        raise SolverNotApplicable(f\"SuperLU could not factor this matrix: {e}\") from e\n    work.setup = time.perf_counter() - t0\n    x = lu.solve(b)\n    work.tri_solves += 2\n    return x, 1, True, work\n\n\ndef sparse_spsolve(A, b):\n    \"\"\"The one-line answer: ``spsolve(A, b)``. What a practitioner types.\"\"\"\n    work = Work()\n    t0 = time.perf_counter()\n    try:\n        x = spla.spsolve(A.tocsc(), b)\n    except RuntimeError as e:\n        raise SolverNotApplicable(f\"spsolve failed: {e}\") from e\n    work.setup = time.perf_counter() - t0\n    work.tri_solves += 2\n    return x, 1, True, work\n\n\ndef ilu_only(A, b):\n    \"\"\"Zero-iteration control: apply the ILU factorization to b and stop.\n\n    This is the baseline that says how much of the proposed method's result came\n    from the preconditioner alone, with no Krylov iteration on top. Without it\n    there is no way to attribute the gain.\n    \"\"\"\n    work = Work()\n    apply_ilu = build_ilu(A, work)\n    if apply_ilu is None:\n        raise SolverNotApplicable(\"no usable ILU factorization for this matrix\")\n    x = apply_ilu(b)\n    work.precond += 1\n    return x, 0, True, work\n\n\ndef ilu_bicgstab(A, b):\n    \"\"\"ILU-preconditioned BiCGSTAB, run unconditionally -- no dispatch.\"\"\"\n    work = Work()\n    apply_ilu = build_ilu(A, work)\n    if apply_ilu is None:\n        raise SolverNotApplicable(\"no usable ILU factorization for this matrix\")\n    x, its, conv, w = bicgstab(A, b, M=apply_ilu)\n    w.setup = work.setup\n    return x, its, conv, w\n\n\ndef ilu_gmres(A, b, restart=30):\n    \"\"\"ILU-preconditioned restarted GMRES: PETSc's default configuration.\"\"\"\n    work = Work()\n    apply_ilu = build_ilu(A, work)\n    if apply_ilu is None:\n        raise SolverNotApplicable(\"no usable ILU factorization for this matrix\")\n    x, its, conv, w = gmres(A, b, M=apply_ilu, restart=restart)\n    w.setup = work.setup\n    return x, its, conv, w\n\n\ndef ilu_krylov_dispatched(A, b):\n    \"\"\"Symmetry-dispatched ILU-preconditioned Krylov: PCG if A is symmetric,\n    BiCGSTAB otherwise.\n\n    This is the method the proposal called APK and claimed as novel. It is the\n    decision rule in Barrett et al., *Templates* (SIAM, 1994), and the default\n    behaviour of PETSc's KSP and MATLAB's backslash dispatch. It is kept in the\n    benchmark as a baseline so the dispatch rule itself can be measured against\n    ``ilu_bicgstab`` and ``ilu_gmres`` running unconditionally -- an earlier\n    probe put dispatch at 51/70 against always-BiCGSTAB's 51/70, i.e. no\n    measurable contribution, and that comparison belongs in the results rather\n    than in a claim.\n\n    Known defect, reported rather than hidden: spilu pivots and permutes, so on\n    a symmetric A the factor is not symmetric and the PCG branch violates CG's\n    preconditioner requirement.\n    \"\"\"\n    work = Work()\n    apply_ilu = build_ilu(A, work)\n    if apply_ilu is None:\n        raise SolverNotApplicable(\"no usable ILU factorization for this matrix\")\n    if _is_symmetric(A):\n        x, its, conv, w = pcg(A, b, M=apply_ilu)\n        if conv and np.all(np.isfinite(x)):\n            w.setup = work.setup\n            return x, its, conv, w\n    x, its, conv, w = bicgstab(A, b, M=apply_ilu)     # fallback and default path\n    w.setup = work.setup\n    return x, its, conv, w\n", "refinement": "\"\"\"Iterative refinement as an orthogonal factor, available to every solver.\n\nIn the first sweep refinement was applied only to the proposed method, and that\nsingle asymmetry produced its headline accuracy result: refined, it reached a\nmedian error of 1.45e-14 against unrefined baselines. The same wrapper placed\naround Gauss-Seidel reaches 5.44e-16 -- about 27 times more accurate. The\naccuracy ranking was measuring the wrapper, not the solver.\n\nSo refinement lives here, outside any solver, and the harness runs every method\nat each pass count in ``config.REFINEMENT_PASSES``. The extra cost is real and\nis counted: each pass is a full additional solve of the residual equation.\n\nThe procedure is the classical one (Wilkinson 1963): solve, form the residual,\nsolve the residual equation with the same method, correct. Working in the same\nprecision throughout, it cannot repair a badly conditioned system, but it does\nrecover the accuracy an early-stopped iterative solve leaves on the table.\n\"\"\"\nimport numpy as np\n\nfrom .iterative_solvers import Work\n\n\ndef refine(solver, A, b, passes=1):\n    \"\"\"Run `solver` on (A, b), then apply `passes` correction steps.\n\n    `solver` is any callable with the project's ``(x, iterations, converged,\n    work)`` contract. Returns the same tuple, with iterations and work summed\n    across the initial solve and every correction, so a refined run is never\n    credited with the cost of an unrefined one.\n    \"\"\"\n    x, iterations, converged, work = solver(A, b)\n    if passes <= 0 or x is None or not np.all(np.isfinite(x)):\n        return x, iterations, converged, work\n\n    total = Work()\n    total.matvecs = work.matvecs\n    total.tri_solves = work.tri_solves\n    total.precond = work.precond\n    total.setup = work.setup\n\n    b_norm = np.linalg.norm(b) or 1.0\n    for _ in range(passes):\n        r = b - A @ x\n        total.matvecs += 1\n        if not np.all(np.isfinite(r)):\n            break\n        # Nothing left to correct. Handing a zero right-hand side to the solver\n        # is not merely wasteful: CG's first search direction is then the zero\n        # vector, p'Ap is exactly 0, and it reports the matrix as indefinite --\n        # so an exact solve would be recorded as a solver failure.\n        if np.linalg.norm(r) <= np.finfo(float).eps * b_norm:\n            break\n        d, its, conv, w = solver(A, r)\n        total.matvecs += w.matvecs\n        total.tri_solves += w.tri_solves\n        total.precond += w.precond\n        total.setup += w.setup\n        iterations += its\n        if d is None or not np.all(np.isfinite(d)):\n            break\n        x_next = x + d\n        if not np.all(np.isfinite(x_next)):\n            break\n        x = x_next\n        converged = converged or conv\n\n    return x, iterations, converged, total\n", "io_utils": "\"\"\"Loading matrices and building the ground-truth (x_true, b) pair.\n\nThe corpus is laid out flat as ``<domain>/<name>.mtx``. An earlier version of\nthis module expected the nested ``<domain>/<name>/<name>.mtx`` shape the pilot\ndataset used, so it silently found nothing in the real corpus -- one of the ways\nthe library and the notebook that produced the results had drifted apart.\nBoth layouts are accepted now.\n\"\"\"\nfrom pathlib import Path\n\nimport numpy as np\nimport scipy.io\nimport scipy.sparse as sp\n\n\ndef load_matrix(mtx_path):\n    \"\"\"Load a Matrix Market file as a real-valued square CSR matrix.\"\"\"\n    A = scipy.io.mmread(str(mtx_path))\n    A = sp.csr_matrix(A, dtype=np.float64)\n    if A.shape[0] != A.shape[1]:\n        raise ValueError(f\"{mtx_path} is not square: {A.shape}\")\n    return A\n\n\ndef make_ground_truth(A, seed=None):\n    \"\"\"Return (x_true, b) with b = A @ x_true, so 'correct' is unambiguous.\n\n    With `seed` unset, x_true is the vector of ones -- deterministic and easy to\n    reason about, but a special vector: it is in the null space of any matrix\n    whose rows sum to zero, and it makes b a plain row-sum, which for a handful\n    of matrices in this corpus is almost entirely cancellation (the worst has\n    ||A.1|| / |||A|.1|| = 3.5e-17, meaning b is essentially rounding noise).\n    Passing a seed draws x_true from N(0,1) instead, which is how the replicate\n    runs check that no result depends on that choice.\n    \"\"\"\n    n = A.shape[0]\n    if seed is None:\n        x_true = np.ones(n, dtype=np.float64)\n    else:\n        x_true = np.random.default_rng(seed).standard_normal(n)\n    return x_true, A @ x_true\n\n\ndef cancellation_ratio(A, x_true):\n    \"\"\"||A x|| / |||A| |x|||: how much of b survives cancellation.\n\n    Near machine epsilon, b carries almost no information about A and no solver\n    can be judged on it. Reported per matrix so those systems can be excluded\n    from accuracy statistics rather than quietly distorting them.\n    \"\"\"\n    num = float(np.linalg.norm(A @ x_true))\n    den = float(np.linalg.norm(abs(A) @ np.abs(x_true)))\n    return num / den if den > 0 else np.nan\n\n\ndef structural_singularity(A):\n    \"\"\"Count all-zero rows and columns.\n\n    Such a matrix has no unique solution, so every solver result on it is\n    meaningless. Screening is also a hard robustness requirement: SuperLU's\n    spilu raises a catchable error on these under scipy 1.17 but exhausts memory\n    and gets the process OS-killed under Kaggle's scipy 1.16.3. Two full runs\n    died this way, both at M40PI_n1. 19 of 930 matrices are affected.\n    \"\"\"\n    zero_rows = int((np.diff(A.indptr) == 0).sum())\n    zero_cols = int((np.diff(A.tocsc().indptr) == 0).sum())\n    return zero_rows, zero_cols\n\n\ndef discover_matrices(dataset_root):\n    \"\"\"Find every .mtx under the corpus root, flat or nested, as one entry each.\"\"\"\n    root = Path(dataset_root)\n    entries = []\n    for domain_dir in sorted(p for p in root.iterdir() if p.is_dir()):\n        for mtx_path in sorted(domain_dir.rglob(\"*.mtx\")):\n            entries.append({\"domain\": domain_dir.name,\n                            \"name\": mtx_path.stem,\n                            \"path\": mtx_path})\n    return entries\n", "spectral": "\"\"\"Spectral radii of the iteration matrices, and hypothesis-class labels.\n\nThis is the quantity the base paper exists to characterise, and the first sweep\nnever computed it. Khrapov and Volkov derive exactly where rho(T) < 1 holds for\n2 and 3 unknowns; an extension of their work that measures convergence\nempirically but never looks at rho has no way to say *why* a method converged.\n\nTwo things are computed here.\n\n**Spectral radius.** For the Jacobi iteration matrix T_J = I - D^-1 A and the\nGauss-Seidel iteration matrix T_GS = -(D+L)^-1 U. A stationary method converges\nfrom any starting vector if and only if rho(T) < 1, so this predicts the\noutcome the benchmark measures, and the two can be compared.\n\n**Hypothesis class.** Which classical theorem, if any, covers the matrix:\n\n* *Strict diagonal dominance* -- both Jacobi and Gauss-Seidel converge.\n* *L-matrix* (positive diagonal, non-positive off-diagonal). With rho(T_J) < 1\n  this is an M-matrix, and Stein and Rosenberg (1948) then give\n  rho(T_GS) < rho(T_J) < 1: Gauss-Seidel converges whenever Jacobi does, and\n  strictly faster. \"Jacobi converges but Gauss-Seidel does not\" is impossible\n  here -- which is why finding zero such cases in real data is not a discovery.\n* *H-matrix* -- rho of the comparison matrix's Jacobi iteration is below 1;\n  both methods converge.\n* *SPD* -- Householder (1958) and John: Gauss-Seidel always converges.\n* *Property A / consistently ordered* -- Young (1950) gives\n  rho(T_GS) = rho(T_J)^2 and an optimal relaxation factor\n  omega* = 2 / (1 + sqrt(1 - rho(T_J)^2)). Detected here by the numerical\n  identity rather than by inspecting the ordering, so it is a proxy.\n\nCoverage of these classes across the corpus is the measurement that explains the\nbenchmark's own headline result, and it is what any referee would ask for first.\n\"\"\"\nimport numpy as np\nimport scipy.sparse as sp\nimport scipy.sparse.linalg as spla\n\nfrom . import config\n\n\ndef _diagonal(A):\n    d = A.diagonal()\n    return None if np.any(np.abs(d) < 1e-14) else d\n\n\ndef jacobi_operator(A):\n    \"\"\"T_J = I - D^-1 A, applied without forming it.\"\"\"\n    d = _diagonal(A)\n    if d is None:\n        return None\n    return spla.LinearOperator(A.shape, matvec=lambda v: v - (A @ v) / d, dtype=np.float64)\n\n\ndef gauss_seidel_operator(A):\n    \"\"\"T_GS = -(D + L)^-1 U, with the triangular factor built once.\"\"\"\n    if _diagonal(A) is None:\n        return None\n    M = sp.tril(A, format=\"csc\")\n    U = sp.triu(A, k=1, format=\"csr\")\n    try:\n        lu = spla.splu(M, permc_spec=\"NATURAL\", diag_pivot_thresh=0.0)\n    except RuntimeError:\n        return None\n    return spla.LinearOperator(A.shape, matvec=lambda v: -lu.solve(U @ v), dtype=np.float64)\n\n\ndef _power_iteration(op, n, rng=None):\n    \"\"\"Estimate the dominant eigenvalue magnitude by repeated application.\n\n    Returns (rho, converged). Reported as an estimate, never as an exact value:\n    with a complex dominant pair the ratio oscillates and settles on the modulus\n    more slowly than for a real dominant eigenvalue.\n    \"\"\"\n    rng = rng or np.random.default_rng(0)\n    v = rng.standard_normal(n)\n    v /= np.linalg.norm(v) or 1.0\n    rho = np.nan\n    for _ in range(config.POWER_ITER_MAX):\n        w = op @ v\n        nw = np.linalg.norm(w)\n        if not np.isfinite(nw):\n            return np.inf, False\n        if nw == 0.0:\n            return 0.0, True\n        prev, rho = rho, nw\n        v = w / nw\n        if np.isfinite(prev) and abs(rho - prev) <= config.POWER_ITER_TOL * max(rho, 1e-30):\n            return float(rho), True\n    return float(rho), False\n\n\ndef dense_iteration_matrix(A, kind=\"jacobi\"):\n    \"\"\"Build the iteration matrix densely, in one shot rather than column by column.\n\n    Applying the operator to n basis vectors in a Python loop costs the same\n    arithmetic but pays n rounds of interpreter and dispatch overhead, which is\n    the difference between a usable sweep over 611 matrices and an unusable one.\n    Both forms below hand the whole right-hand side over at once instead.\n    \"\"\"\n    d = _diagonal(A)\n    if d is None:\n        return None\n    n = A.shape[0]\n    if kind == \"jacobi\":\n        return np.eye(n) - A.toarray() / d[:, None]\n\n    M = sp.tril(A, format=\"csc\")\n    U = sp.triu(A, k=1, format=\"csr\")\n    try:\n        lu = spla.splu(M, permc_spec=\"NATURAL\", diag_pivot_thresh=0.0)\n    except RuntimeError:\n        return None\n    return -lu.solve(U.toarray())          # splu.solve takes all columns at once\n\n\ndef spectral_radius(A, kind=\"jacobi\", exact_cap=None):\n    \"\"\"rho of an iteration matrix. Returns (rho, how).\n\n    `how` records which route produced the number -- \"exact_eig\" from a dense\n    eigendecomposition, \"arpack\" or \"power_iteration\" otherwise -- so results\n    computed by different routes are never silently pooled.\n    \"\"\"\n    exact_cap = config.SPECTRAL_EXACT_CAP if exact_cap is None else exact_cap\n    n = A.shape[0]\n    op = jacobi_operator(A) if kind == \"jacobi\" else gauss_seidel_operator(A)\n    if op is None:\n        return np.nan, \"not_applicable\"\n\n    if n <= exact_cap:\n        T = dense_iteration_matrix(A, kind)\n        try:\n            if T is not None:\n                return float(np.max(np.abs(np.linalg.eigvals(T)))), \"exact_eig\"\n        except (np.linalg.LinAlgError, MemoryError):\n            pass\n\n    try:\n        vals = spla.eigs(op, k=1, which=\"LM\", return_eigenvectors=False,\n                         maxiter=config.POWER_ITER_MAX * 10, tol=config.POWER_ITER_TOL)\n        return float(np.abs(vals[0])), \"arpack\"\n    except Exception:\n        rho, ok = _power_iteration(op, n)\n        return rho, \"power_iteration\" if ok else \"power_iteration_unconverged\"\n\n\ndef iterations_to_tolerance(rho, tol=None):\n    \"\"\"How many iterations rho^k <= tol needs. Infinite when rho >= 1.\n\n    The asymptotic rate only, ignoring the transient, so it is a lower bound on\n    what a real run costs rather than a prediction of it.\n    \"\"\"\n    tol = config.TOLERANCE if tol is None else tol\n    if not np.isfinite(rho) or rho >= 1.0:\n        return np.inf\n    if rho <= 0.0:\n        return 1.0\n    return float(np.log(tol) / np.log(rho))\n\n\ndef convergence_verdict(rho, tol=None, budget=None):\n    \"\"\"Three-way prediction, where the textbook criterion gives only two.\n\n    Convergence theory asks a yes/no question -- is rho < 1 -- but a matrix can\n    satisfy it and still be useless. nos7 in this corpus has rho(T_J) =\n    0.999999984536822: strictly below 1, so Jacobi provably converges, and it\n    would take about 1.19e9 iterations to reach 1e-8 against an iteration cap of\n    10,000. Reporting that as a prediction failure would be wrong, and reporting\n    it as \"converges\" would be misleading. It is a third case.\n\n    A fourth case has to be kept apart from these three. When the diagonal carries a\n    zero the iteration matrix does not exist, so rho is NaN -- and \"the method is\n    undefined here\" is not the same statement as \"it would diverge\". Conflating them put\n    491 systems into the diverging column for Jacobi, which is most of that column.\n\n    Returns \"not_applicable\", \"diverges\", \"too_slow\", or \"converges\".\n    \"\"\"\n    budget = config.MAX_ITERATIONS if budget is None else budget\n    if rho is None or (isinstance(rho, float) and np.isnan(rho)):\n        return \"not_applicable\"\n    if not np.isfinite(rho) or rho >= 1.0:      # +inf is genuine divergence\n        return \"diverges\"\n    return \"converges\" if iterations_to_tolerance(rho, tol) <= budget else \"too_slow\"\n\n\ndef comparison_matrix(A):\n    \"\"\"M(A): |a_ii| on the diagonal, -|a_ij| off it. A is an H-matrix exactly\n    when M(A) is an M-matrix, which is what the H-matrix test below checks.\"\"\"\n    M = -abs(A).tolil()\n    M.setdiag(np.abs(A.diagonal()))\n    return M.tocsr()\n\n\ndef is_strictly_diagonally_dominant(A):\n    d = np.abs(A.diagonal())\n    off = np.asarray(abs(A).sum(axis=1)).ravel() - d\n    return bool(np.all(d > off))\n\n\ndef is_weakly_diagonally_dominant(A):\n    d = np.abs(A.diagonal())\n    off = np.asarray(abs(A).sum(axis=1)).ravel() - d\n    return bool(np.all(d >= off) and np.any(d > off))\n\n\ndef is_l_matrix(A):\n    d = A.diagonal()\n    if np.any(d <= 0):\n        return False\n    off = A - sp.diags(d)\n    return bool(off.nnz == 0 or off.max() <= 0)\n\n\ndef is_symmetric(A, tol=1e-8):\n    diff = abs(A - A.T)\n    return bool(diff.nnz == 0 or diff.max() <= tol * max(abs(A).max(), 1.0))\n\n\ndef is_spd(A):\n    \"\"\"Symmetric with a successful sparse Cholesky-equivalent factorization.\"\"\"\n    if not is_symmetric(A):\n        return False\n    try:\n        vals = spla.eigsh(A.astype(np.float64), k=1, which=\"SA\",\n                          return_eigenvectors=False, maxiter=5000)\n        return bool(vals[0] > 0)\n    except Exception:\n        try:\n            np.linalg.cholesky(A.toarray())\n            return True\n        except (np.linalg.LinAlgError, MemoryError, ValueError):\n            return False\n\n\ndef classify(A, rho_jacobi=None, rho_gs=None):\n    \"\"\"Label one matrix with every hypothesis class it satisfies.\n\n    Returns the individual flags plus `hypothesis_class`, a single label chosen\n    by the precedence below for reporting convenience. The flags are the real\n    output -- a matrix can belong to several classes at once, and collapsing\n    that to one label loses information.\n    \"\"\"\n    flags = {\n        \"strict_diag_dominant\": is_strictly_diagonally_dominant(A),\n        \"weak_diag_dominant\": is_weakly_diagonally_dominant(A),\n        \"l_matrix\": is_l_matrix(A),\n        \"symmetric\": is_symmetric(A),\n        \"spd\": is_spd(A),\n    }\n\n    if rho_jacobi is None:\n        rho_jacobi, _ = spectral_radius(A, \"jacobi\")\n    if rho_gs is None:\n        rho_gs, _ = spectral_radius(A, \"gauss_seidel\")\n\n    # H-matrix: rho of the comparison matrix's Jacobi iteration below 1.\n    rho_comparison, _ = spectral_radius(comparison_matrix(A), \"jacobi\")\n    flags[\"h_matrix\"] = bool(np.isfinite(rho_comparison) and rho_comparison < 1.0)\n\n    # M-matrix: an L-matrix whose Jacobi iteration converges. This is the class\n    # in which Stein-Rosenberg forbids \"Jacobi converges, Gauss-Seidel does not\".\n    flags[\"m_matrix\"] = bool(flags[\"l_matrix\"] and np.isfinite(rho_jacobi) and rho_jacobi < 1.0)\n\n    # Property A proxy: Young's rho(T_GS) = rho(T_J)^2 holding numerically.\n    flags[\"property_a_proxy\"] = bool(\n        np.isfinite(rho_jacobi) and np.isfinite(rho_gs) and rho_jacobi > 0\n        and abs(rho_gs - rho_jacobi ** 2) <= 1e-3 * max(rho_jacobi ** 2, 1e-12)\n    )\n\n    for label in (\"m_matrix\", \"spd\", \"strict_diag_dominant\", \"h_matrix\",\n                  \"l_matrix\", \"weak_diag_dominant\", \"property_a_proxy\"):\n        if flags[label]:\n            primary = label\n            break\n    else:\n        primary = \"none\"\n\n    return {**flags,\n            \"rho_jacobi\": rho_jacobi,\n            \"rho_gauss_seidel\": rho_gs,\n            \"rho_comparison\": rho_comparison,\n            \"hypothesis_class\": primary,\n            # Theory's binary answer, kept because it is what the classical\n            # theorems actually state ...\n            \"jacobi_converges_predicted\": bool(np.isfinite(rho_jacobi) and rho_jacobi < 1.0),\n            \"gs_converges_predicted\": bool(np.isfinite(rho_gs) and rho_gs < 1.0),\n            # ... and the three-way verdict, which is what the benchmark can\n            # observe within a finite iteration budget.\n            \"jacobi_verdict\": convergence_verdict(rho_jacobi),\n            \"gs_verdict\": convergence_verdict(rho_gs),\n            \"jacobi_iterations_needed\": iterations_to_tolerance(rho_jacobi),\n            \"gs_iterations_needed\": iterations_to_tolerance(rho_gs),\n            \"optimal_omega\": (2.0 / (1.0 + np.sqrt(1.0 - rho_jacobi ** 2))\n                              if np.isfinite(rho_jacobi) and rho_jacobi < 1.0 else np.nan)}\n", "reordering": "\"\"\"Convergence-oriented diagonal selection.\n\nA stationary method divides by ``a_ii``, so its behaviour depends entirely on which\nentries end up on the diagonal -- and that is decided by the order the rows happened to\narrive in, which came from a mesh numbering or a netlist, not from any thought about\nconvergence. This module treats that as a choice to be optimised.\n\n**Why permutation is the only lever.** For the Jacobi iteration matrix\n``T_J = I - D^-1 A``:\n\n* Row scaling ``A' = RA`` gives ``D' = RD`` and ``T_J' = I - (RD)^-1 RA = T_J``. The\n  spectrum is *unchanged*, exactly.\n* Column scaling ``A' = AC`` gives ``T_J' = C^-1 (I - D^-1 A) C``, a similarity\n  transform. The spectrum is unchanged again.\n* A symmetric permutation ``PAP^T`` leaves the spectrum of ``T_J`` invariant, and on\n  real matrices moves ``rho(T_GS)`` only in the fifth decimal (measured with both RCM\n  and random orderings).\n\nSo scaling and symmetric reordering cannot help. Only a **row permutation** can, because\nonly it changes which entries make up ``D``.\n\n**Why not just use MC64.** MC64 (Duff & Koster) also produces a row permutation, but it\nmaximises the *product* of the diagonal magnitudes -- an objective designed for pivot\nstability in a direct factorization. Stationary convergence needs something else: a small\nrow ratio ``sum_{j != i} |a_ij| / |a_ii|``, since all such ratios below 1 means strict\ndiagonal dominance and guarantees convergence. Optimising the right objective gives a\ndifferent permutation and a different outcome: on ``odepa400`` MC64 leaves the matrix\nuntouched at rho(T_GS) = 1.0001, while the dominance objective reaches 0.495 and\nGauss-Seidel converges.\n\n**The solution is untouched.** ``A' x = PAx = Pb = b'``, so a row permutation changes the\nsplitting without changing what solves the system. Nothing needs un-permuting afterwards.\n\nEverything here works on the sparse pattern: the ratio matrix has exactly the nonzeros of\nA, so cost is O(nnz), not O(n^2).\n\"\"\"\nimport time\n\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\n\nOBJECTIVES = (\"bottleneck\", \"minsum\", \"mc64\", \"best\", \"none\")\n\n#: Candidates the \"best\" objective chooses among. No single one dominates: on\n#: ``odepa400`` bottleneck reaches rho(T_GS) = 0.495 where min-sum gives 1.265 and MC64\n#: leaves it at 1.0001; on ``d_ss`` bottleneck is the worst of the three at 29.6 against\n#: MC64's 1.83. Since each permutation costs milliseconds, the honest method computes\n#: all of them and picks by direct spectral estimate.\nPORTFOLIO = (\"none\", \"mc64\", \"minsum\", \"bottleneck\")\n\n#: Every assignment is solved densely up to this size. Set above the corpus maximum of\n#: n = 10,000 deliberately, because the earlier cap of 5,000 was the bug rather than the\n#: safeguard: it left the sparse routines in play for the largest matrices, and rw5151\n#: (n = 5,151, 151 past the cap) then hung the bottleneck search.\n#:\n#: Measured cost of the dense route at n = 10,000, an 800 MB array:\n#:   min-cost assignment (MC64, min-sum)   7.9 s, once per matrix\n#:   0/1 feasibility test (bottleneck)     0.5 s, about ten times per matrix\n#: Kaggle offers roughly 30 GB, so the array is affordable and the time is not the\n#: bottleneck. Past this size the sparse fallback returns, with the risk that implies.\nDENSE_ASSIGNMENT_CAP = 12000\n\n\ndef row_ratios(A):\n    \"\"\"Sparse R with A's pattern: ``R[i,j]`` is what row i's Jacobi ratio becomes if\n    ``a_ij`` is chosen as its diagonal, i.e. ``(sum_k |a_ik| - |a_ij|) / |a_ij|``.\n\n    A value below 1 means that row would be diagonally dominant under this choice.\n    \"\"\"\n    M = abs(A).tocsr()\n    rowsum = np.asarray(M.sum(axis=1)).ravel()\n    R = M.copy().astype(np.float64)\n    counts = np.diff(R.indptr)\n    R.data = (np.repeat(rowsum, counts) - M.data) / M.data\n    return R\n\n\ndef worst_row_ratio(A):\n    \"\"\"``max_i (sum_{j != i} |a_ij|) / |a_ii|`` for the diagonal ``A`` currently has.\n\n    Below 1 is strict diagonal dominance, which guarantees BOTH Jacobi and Gauss-Seidel\n    converge -- so this one number says whether a permutation bought a convergence\n    guarantee or merely a better-looking diagonal. Infinite when any diagonal entry is\n    zero, which is the case the stationary methods are undefined on.\n\n    O(nnz), read from the stored diagonal directly. It must NOT go through\n    ``row_ratios(A).diagonal()``: a structurally absent a_ii has no entry there, so that\n    route reports a ratio of 0 -- perfect dominance -- for exactly the matrices that have\n    no usable diagonal at all.\n    \"\"\"\n    M = abs(A).tocsr()\n    rowsum = np.asarray(M.sum(axis=1)).ravel()\n    d = np.abs(A.diagonal())\n    with np.errstate(divide=\"ignore\", invalid=\"ignore\"):\n        r = (rowsum - d) / d\n    r[d == 0] = np.inf\n    return float(np.max(r)) if r.size else np.inf\n\n\ndef _perm_from_matching(match_rows_to_cols, n):\n    \"\"\"Row r assigned column c must sit at position c, so ``p[c] = r`` and ``A[p, :]``\n    puts the chosen entry on the diagonal.\"\"\"\n    p = np.empty(n, dtype=int)\n    p[match_rows_to_cols] = np.arange(n)\n    return p\n\n\ndef bottleneck_permutation(A):\n    \"\"\"Minimise the WORST row ratio: ``min_pi max_i R[i, pi(i)]``.\n\n    Bottleneck assignment, solved by binary search on the threshold -- keep only edges\n    with ratio <= t and ask whether a perfect matching still exists. Returns\n    ``(perm, worst_ratio)``, or ``(None, inf)`` if no perfect matching exists at all\n    (then no permutation can give a nonzero diagonal).\n    \"\"\"\n    n = A.shape[0]\n    if n > DENSE_ASSIGNMENT_CAP:\n        return None, np.inf          # see DENSE_ASSIGNMENT_CAP; the portfolio copes\n\n    R = row_ratios(A)\n    if R.nnz == 0:\n        return None, np.inf\n\n    candidates = np.unique(R.data)\n\n    # Two prunings, both needed at scale. Each binary-search step runs a matching over\n    # the whole pattern, so on TSC_OPF_1047 (n = 8,140, nnz = 2.0M) the naive search\n    # over every distinct ratio took 103 seconds.\n    #\n    # First: MC64 is cheap and always yields a valid permutation, so the worst ratio it\n    # achieves is an upper bound on the optimum. Everything above it can be discarded\n    # before the search starts.\n    mc_perm, _ = mc64_permutation(A)\n    if mc_perm is not None:\n        upper = worst_row_ratio(A[mc_perm, :])\n        if np.isfinite(upper):\n            kept = candidates[candidates <= upper]\n            if kept.size:\n                candidates = kept\n\n    # Second: cap the search to a bounded number of steps. Above this many distinct\n    # ratios the thresholds are taken as quantiles, so the result is the optimum to\n    # within one quantile rather than exactly -- a permutation good enough to rank,\n    # which is all the selector needs.\n    if candidates.size > 1024:\n        candidates = np.unique(np.quantile(candidates, np.linspace(0, 1, 1024)))\n\n    dense = R.toarray()\n    pattern = abs(A).toarray() > 0\n\n    def feasible(threshold):\n        \"\"\"Does a perfect matching exist using only entries at or below `threshold`?\n\n        Always dense. maximum_bipartite_matching was the third scipy routine in this\n        module to misbehave on real inputs: on bcsstk19 -- n = 817, 6,853 nonzeros -- a\n        single call took 7.2 seconds on a 3,764-edge subgraph, and the search makes ten\n        such calls. It is slowest precisely when a matching does exist, which is the case\n        the search spends most of its time in. The dense form answers the same question\n        by assignment with 0/1 costs: a total of zero means every row found an allowed\n        column.\n        \"\"\"\n        allowed = pattern & (dense <= threshold)\n        cost = np.where(allowed, 0.0, 1.0)\n        rows, cols = linear_sum_assignment(cost)\n        if cost[rows, cols].sum() > 0:\n            return False, None\n        m = np.empty(n, dtype=int)\n        m[rows] = cols\n        return True, m\n\n    lo, hi, best = 0, candidates.size - 1, None\n    while lo <= hi:\n        mid = (lo + hi) // 2\n        ok, m = feasible(candidates[mid])\n        if ok:\n            best = (np.asarray(m).copy(), candidates[mid])\n            hi = mid - 1\n        else:\n            lo = mid + 1\n    if best is None:\n        return None, np.inf\n    m, worst = best\n    return _perm_from_matching(m, n), float(worst)\n\n\ndef minsum_permutation(A):\n    \"\"\"Minimise the TOTAL of the chosen row ratios: ``min_pi sum_i R[i, pi(i)]``.\n\n    **This is MC64 under another name.** Since ``1 + R[i,j] = rowsum_i / |a_ij|``,\n\n        sum_i log(1 + R[i,p(i)])  =  sum_i log(rowsum_i)  -  sum_i log|a_{i,p(i)}|\n\n    and the first term does not depend on the permutation, so minimising the left side is\n    exactly maximising ``sum log|a_ii|`` -- MC64's objective. Verified on 193 of 193\n    random matrices, both reaching an identical objective value.\n\n    It is kept because the full run reports it separately, and because the equivalence is\n    worth stating: it was introduced as \"the natural contrast to bottleneck\" and is not a\n    contrast at all. The portfolio has two distinct objectives, not three.\n    \"\"\"\n    n = A.shape[0]\n    if n > DENSE_ASSIGNMENT_CAP:\n        return None, np.inf          # see DENSE_ASSIGNMENT_CAP; the portfolio copes\n\n    R = row_ratios(A)\n    if R.nnz == 0:\n        return None, np.inf\n\n    # Minimise sum(log(1 + ratio)), i.e. the PRODUCT of (1 + ratio) rather than the raw\n    # sum. A raw sum is dominated by whichever single row has the largest ratio -- on\n    # nnc261 those span 0 to 3.8e10 -- which would make \"min-sum\" nearly the bottleneck\n    # objective it exists to contrast with.\n    mask = abs(A).toarray() > 0\n    return _dense_assignment(np.log1p(R.toarray()), mask, n)\n\n\ndef _dense_assignment(weights, mask, n):\n    \"\"\"Solve a minimum-cost perfect assignment densely.\n\n    ``weights`` holds the cost of every real edge and ``mask`` says which entries are\n    real. Absent edges get a penalty larger than any complete assignment of real ones, so\n    a chosen penalty edge means no perfect matching exists over the real entries.\n\n    Dense rather than ``min_weight_full_bipartite_matching`` because that routine is not\n    dependable here. It hung outright on nnc261 under raw ratios and on west0067 -- 67x67,\n    294 nonzeros -- under a log transform, and merely crawled elsewhere: 9.7 seconds on\n    oscil_dcop_23 at n = 430, which the dense solver finishes in 6 milliseconds, a factor\n    of 1,600. One of those hangs cost a twelve-hour Kaggle session that completed 30\n    matrices of 930. A hang cannot be interrupted from Python, so the fix has to be\n    avoiding the routine rather than detecting the hang.\n    \"\"\"\n    W = np.full((n, n), np.inf)\n    W[mask] = weights[mask]\n    finite = W[np.isfinite(W)]\n    if finite.size == 0:\n        return None, np.inf\n    penalty = (finite.max() + 1.0) * n + 1.0\n    W[~np.isfinite(W)] = penalty\n    rows, cols = linear_sum_assignment(W)\n    if np.any(W[rows, cols] >= penalty):\n        return None, np.inf          # no perfect matching over the real entries\n    return _perm_from_matching(cols[np.argsort(rows)], n), float(W[rows, cols].sum())\n\n\ndef mc64_permutation(A):\n    \"\"\"Baseline: maximise the product of |diagonal| entries, as MC64 does.\n\n    Equivalent to minimising ``sum -log|a_ij|``. This is the established tool, built for\n    pivot stability rather than for convergence, and it is the comparison the method has\n    to beat.\n    \"\"\"\n    n = A.shape[0]\n    M = abs(A).tocsr().astype(np.float64)\n    if M.nnz == 0:\n        return None, np.inf\n\n    if n <= DENSE_ASSIGNMENT_CAP:\n        d = M.toarray()\n        mask = d > 0\n        with np.errstate(divide=\"ignore\"):\n            C = -np.log(d, out=np.full_like(d, -np.inf), where=mask)\n        C[mask] -= C[mask].min() - 1.0          # strictly positive weights\n        return _dense_assignment(C, mask, n)\n\n    return None, np.inf              # see DENSE_ASSIGNMENT_CAP; the portfolio copes\n\n\ndef apply_permutation(A, b, p):\n    \"\"\"``A' = P A``, ``b' = P b``. The solution x is unchanged, so no un-permuting.\"\"\"\n    return A[p, :].tocsr(), b[p]\n\n\ndef select_diagonal(A, b=None, objective=\"bottleneck\"):\n    \"\"\"Choose the diagonal, returning ``(A', b', info)``.\n\n    With ``objective=\"none\"`` the matrix is returned untouched, which is the control\n    condition for the ablation.\n    \"\"\"\n    n = A.shape[0]\n    info = {\"objective\": objective, \"permuted\": False, \"cost\": np.nan,\n            \"zero_diagonal_before\": int((np.abs(A.diagonal()) < 1e-14).sum()),\n            \"worst_ratio_before\": worst_row_ratio(A)}\n\n    if objective == \"none\":\n        return A, b, info\n\n    if objective == \"best\":\n        return _select_best(A, b, info)\n\n    solver = {\"bottleneck\": bottleneck_permutation,\n              \"minsum\": minsum_permutation,\n              \"mc64\": mc64_permutation}[objective]\n    p, cost = solver(A)\n    if p is None:\n        info[\"note\"] = \"no perfect matching: no permutation can fill the diagonal\"\n        return A, b, info\n\n    A2 = A[p, :].tocsr()\n    b2 = None if b is None else b[p]\n    info.update(permuted=not np.array_equal(p, np.arange(n)), cost=cost,\n                zero_diagonal_after=int((np.abs(A2.diagonal()) < 1e-14).sum()),\n                worst_ratio_after=worst_row_ratio(A2))\n    return A2, b2, info\n\n\ndef _estimate_rho_gs(A, iters=40):\n    \"\"\"Cheap power-iteration estimate of rho(T_GS), for ranking candidates only.\n\n    Deliberately not the exact eigendecomposition: this runs once per candidate inside\n    the solver, so it has to cost far less than solving. Ranking needs the ordering to be\n    right, not the value.\n    \"\"\"\n    from . import spectral\n    op = spectral.gauss_seidel_operator(A)\n    if op is None:\n        return np.inf\n    v = np.random.default_rng(0).standard_normal(A.shape[0])\n    nv = np.linalg.norm(v)\n    if nv == 0:\n        return np.inf\n    v /= nv\n    rho = np.inf\n    for _ in range(iters):\n        w = op @ v\n        nw = np.linalg.norm(w)\n        if not np.isfinite(nw):\n            return np.inf\n        if nw == 0.0:\n            return 0.0\n        rho, v = nw, w / nw\n    return float(rho)\n\n\ndef _select_best(A, b, info):\n    \"\"\"Try every candidate permutation and keep the one with the smallest estimated\n    rho(T_GS). Strictly at least as good as any fixed choice, because \"no permutation\"\n    is itself one of the candidates.\"\"\"\n    best = (np.inf, None, None, \"none\")\n    tried, ratios = {}, {}\n    for name in PORTFOLIO:\n        if name == \"none\":\n            cand = A\n        else:\n            p, _ = {\"mc64\": mc64_permutation, \"minsum\": minsum_permutation,\n                    \"bottleneck\": bottleneck_permutation}[name](A)\n            if p is None:\n                continue\n            cand = A[p, :].tocsr()\n        # Every candidate's worst row ratio, recorded whether or not it wins. This is\n        # what lets the write-up compare the objectives on the quantity the convergence\n        # guarantee is stated in, without paying for a fourth condition in the sweep --\n        # the permutations are already computed here.\n        ratios[name] = worst_row_ratio(cand)\n        if np.any(np.abs(cand.diagonal()) < 1e-14):\n            tried[name] = np.inf                 # still undefined, unusable\n            continue\n        rho = _estimate_rho_gs(cand)\n        tried[name] = rho\n        if rho < best[0]:\n            best = (rho, cand, None if name == \"none\" else p, name)\n\n    rho, cand, p, name = best\n    info.update(chosen=name, estimated_rho_gs=rho, candidates=tried, ratios=ratios,\n                permuted=name != \"none\")\n    if cand is None:\n        info[\"note\"] = \"no candidate produced a usable diagonal\"\n        return A, b, info\n    info[\"zero_diagonal_after\"] = 0\n    info[\"worst_ratio_after\"] = worst_row_ratio(cand)\n    return cand, (b if p is None or b is None else b[p]), info\n\n\ndef make_solver(base_solver, objective=\"bottleneck\"):\n    \"\"\"Wrap any stationary solver so it selects its diagonal first.\n\n    The returned callable keeps the project's ``(x, iterations, converged, work)``\n    contract, so it drops straight into the benchmark harness alongside every other\n    method.\n    \"\"\"\n    def solve(A, b):\n        t0 = time.perf_counter()\n        A2, b2, _ = select_diagonal(A, b, objective=objective)\n        setup = time.perf_counter() - t0\n        x, iters, converged, work = base_solver(A2, b2)\n        # The permutation is part of this method's cost, not a free gift from outside.\n        # A wrapper that hides its own setup wins on cost by bookkeeping.\n        if work is not None and hasattr(work, \"setup\"):\n            work.setup += setup\n        return x, iters, converged, work\n    solve.__name__ = f\"{base_solver.__name__}_{objective}\"\n    return solve\n", "benchmark": "\"\"\"The benchmark harness.\n\nFor every matrix in the corpus and every method defined on it, run the solve and\nrecord what happened. Three rules hold throughout, and each exists because the\nfirst sweep broke it:\n\n1. **One judge.** No solver decides its own outcome. Every result goes through\n   :func:`solvebench.metrics.score`, which measures the residual from A, x and b\n   and applies one tolerance to all methods.\n2. **Every matrix is accounted for.** A matrix that is skipped, unloadable or\n   structurally singular still produces a row, with a reason code, for every\n   method. Denominators reconcile to the corpus size by construction rather than\n   by hoping nothing was dropped.\n3. **Refinement is a factor, not a feature.** Every method is run at each\n   refinement pass count, so no method's accuracy is credited to a wrapper the\n   others did not get.\n\"\"\"\nimport time\nimport warnings\n\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\nimport scipy.sparse.linalg as spla\n\nfrom . import config, direct_solvers as direct, iterative_solvers as it\nfrom . import reordering\nfrom . import metrics, reference_solvers as ref, spectral\nfrom .direct_solvers import SolverNotApplicable\nfrom .io_utils import (cancellation_ratio, discover_matrices, load_matrix,\n                       make_ground_truth, structural_singularity)\nfrom .refinement import refine\n\n\nclass Method:\n    \"\"\"One benchmarked method and the facts the harness needs about it.\"\"\"\n\n    def __init__(self, name, fn, family, dense=False, capped=False):\n        self.name = name\n        self.fn = fn\n        self.family = family\n        self.dense = dense      # takes a dense array rather than a sparse matrix\n        self.capped = capped    # subject to DIRECT_SIZE_CAP\n\n    def __repr__(self):\n        return f\"<Method {self.name}>\"\n\n\ndef _wrap_dense(fn):\n    \"\"\"Adapt a hand-written direct solver to the common 4-tuple contract.\"\"\"\n    def call(A_dense, b):\n        x, steps = fn(A_dense, b)\n        return x, steps, None, it.Work()      # None: it makes no convergence claim\n    return call\n\n\n#: Every method in the benchmark. The hand-written direct solvers are labelled\n#: \"direct-handwritten\" and are pedagogical implementations validated against\n#: LAPACK, not performance competitors -- they are pure-Python O(n^3) and about\n#: 50-100x slower than a library call, which is why they alone carry a size cap.\nMETHODS = [\n    Method(\"Gauss elimination\", _wrap_dense(direct.gauss_elimination), \"direct-handwritten\", dense=True, capped=True),\n    Method(\"Gauss-Jordan\", _wrap_dense(direct.gauss_jordan), \"direct-handwritten\", dense=True, capped=True),\n    Method(\"LU\", _wrap_dense(direct.lu_solve), \"direct-handwritten\", dense=True, capped=True),\n    Method(\"Cholesky\", _wrap_dense(direct.cholesky_solve), \"direct-handwritten\", dense=True, capped=True),\n\n    Method(\"spsolve (SuperLU)\", ref.sparse_spsolve, \"direct-library\"),\n    Method(\"splu (SuperLU)\", ref.sparse_lu, \"direct-library\"),\n\n    Method(\"Jacobi\", it.jacobi, \"stationary\"),\n    Method(\"Gauss-Seidel\", it.gauss_seidel, \"stationary\"),\n    Method(\"SOR\", it.sor, \"stationary\"),\n\n    Method(\"Conjugate Gradient\", it.conjugate_gradient, \"krylov\"),\n    Method(\"BiCGSTAB\", it.bicgstab, \"krylov\"),\n    Method(\"GMRES(30)\", lambda A, b: it.gmres(A, b, restart=30), \"krylov\"),\n\n    Method(\"ILU only\", ref.ilu_only, \"preconditioned\"),\n    Method(\"ILU-BiCGSTAB\", ref.ilu_bicgstab, \"preconditioned\"),\n    Method(\"ILU-GMRES(30)\", ref.ilu_gmres, \"preconditioned\"),\n    Method(\"ILU-Krylov (dispatched)\", ref.ilu_krylov_dispatched, \"preconditioned\"),\n]\n\n#: The proposed pipeline, and the arms needed to attribute its effect.\n#:\n#: Two preprocessing steps sit in front of unmodified solvers:\n#:\n#:   1. choose the diagonal by assignment  -- makes D invertible, and makes an\n#:      incomplete factorization constructible, on matrices where neither was true\n#:   2. choose omega from an estimate of rho(T_J) -- replaces the fixed 1.25\n#:\n#: Both are measured separately as well as together, because \"the pipeline helps\" is\n#: not a result: which step helps, and by how much, is. Baselines for every row here\n#: already exist in the main sweep, so this list is run on its own rather than by\n#: repeating all sixteen methods.\n#:\n#: Step 1 is not offered to the direct or unpreconditioned-Krylov families: they never\n#: divide by a diagonal entry, so a row permutation changes their arithmetic without\n#: addressing anything they were failing on.\nPIPELINE_METHODS = [\n    # step 2 alone -- isolates the relaxation factor from the reordering\n    Method(\"SOR (adaptive w)\", it.sor_adaptive, \"pipeline\"),\n\n    # step 1 alone, on the methods that need a usable diagonal to exist at all\n    Method(\"Jacobi + reordered\", reordering.make_solver(it.jacobi, \"best\"), \"pipeline\"),\n    Method(\"Gauss-Seidel + reordered\",\n           reordering.make_solver(it.gauss_seidel, \"best\"), \"pipeline\"),\n    Method(\"SOR + reordered\", reordering.make_solver(it.sor, \"best\"), \"pipeline\"),\n\n    # both steps\n    Method(\"SOR + pipeline\", reordering.make_solver(it.sor_adaptive, \"best\"), \"pipeline\"),\n\n    # step 1 in front of the preconditioned family: ILU fails to build on 166 matrices\n    # and a zero on the diagonal is what closes its last fallback\n    Method(\"ILU-BiCGSTAB + reordered\",\n           reordering.make_solver(ref.ilu_bicgstab, \"best\"), \"pipeline\"),\n    Method(\"ILU-Krylov + reordered\",\n           reordering.make_solver(ref.ilu_krylov_dispatched, \"best\"), \"pipeline\"),\n]\n\nPIPELINE_NAMES = [m.name for m in PIPELINE_METHODS]\n\nMETHOD_NAMES = [m.name for m in METHODS]\n\n\ndef condition_number(A, A_dense, n):\n    \"\"\"Exact via SVD where affordable, a 1-norm estimate otherwise.\n\n    The route is returned alongside the value: an exact condition number and an\n    estimate are not the same measurement and must not be pooled silently.\n    \"\"\"\n    with warnings.catch_warnings():\n        warnings.simplefilter(\"ignore\")\n        if A_dense is not None and n <= config.COND_EXACT_CAP:\n            try:\n                return float(np.linalg.cond(A_dense)), \"exact_svd\"\n            except (np.linalg.LinAlgError, MemoryError):\n                pass\n        try:\n            lu = spla.splu(A.tocsc())\n            inv_norm = spla.onenormest(spla.LinearOperator(\n                A.shape, matvec=lu.solve, rmatvec=lambda v: lu.solve(v, \"T\")))\n            return float(spla.onenormest(A) * inv_norm), \"estimate_1norm\"\n        except Exception:\n            return float(\"inf\"), \"failed\"\n\n\ndef _row(base, method, passes, **extra):\n    return {**base, \"method\": method.name, \"family\": method.family,\n            \"refinement_passes\": passes, **extra}\n\n\ndef run_one_matrix(domain, name, mtx_path, refinement_passes=(0, 1),\n                   with_spectral=True, seed=None, verbose=True, methods=None):\n    \"\"\"Benchmark a set of methods on one matrix. Returns (result_rows, spectral_row).\n\n    ``methods`` defaults to the sixteen baseline methods. Passing ``PIPELINE_METHODS``\n    runs the proposed pipeline instead, on the same corpus and through the same scoring,\n    so the two are directly comparable without repeating the baselines that are already\n    measured.\n    \"\"\"\n    methods = METHODS if methods is None else methods\n    try:\n        A = load_matrix(mtx_path)\n    except Exception as e:\n        base = {\"domain\": domain, \"matrix\": name, \"n\": np.nan, \"nnz\": np.nan}\n        rows = [_row(base, m, p, **metrics.blank(\"load_failed\", f\"{type(e).__name__}: {e}\"))\n                for m in methods for p in refinement_passes]\n        return rows, {\"domain\": domain, \"matrix\": name, \"status\": \"load_failed\"}\n\n    n = A.shape[0]\n    base = {\"domain\": domain, \"matrix\": name, \"n\": n, \"nnz\": A.nnz,\n            \"density\": A.nnz / (n * n)}\n\n    zero_rows, zero_cols = structural_singularity(A)\n    if zero_rows or zero_cols:\n        if verbose:\n            print(f\"    structurally singular ({zero_rows} zero rows, {zero_cols} zero cols)\"\n                  \" -- no unique solution, all methods skipped\")\n        note = f\"{zero_rows} zero rows, {zero_cols} zero cols\"\n        rows = [_row(base, m, p, **metrics.blank(metrics.STATUS_SINGULAR, note))\n                for m in methods for p in refinement_passes]\n        return rows, {**base, \"status\": metrics.STATUS_SINGULAR}\n\n    x_true, b = make_ground_truth(A, seed=seed)\n    b_norm = np.linalg.norm(b) or 1.0\n    xt_norm = np.linalg.norm(x_true) or 1.0\n\n    needs_dense = n <= max(config.DIRECT_SIZE_CAP, config.COND_EXACT_CAP)\n    A_dense = A.toarray() if needs_dense else None\n\n    cond, cond_how = condition_number(A, A_dense, n)\n    base.update(condition_number=cond, condition_method=cond_how,\n                ill_conditioned=bool(cond > config.ILL_CONDITIONED),\n                cancellation_ratio=cancellation_ratio(A, x_true))\n    if verbose:\n        print(f\"    n={n:,} nnz={A.nnz:,} cond={cond:.2e} ({cond_how})\")\n\n    rows = []\n    for m in methods:\n        operand = A_dense if m.dense else A\n        for passes in refinement_passes:\n            if m.capped and n > config.DIRECT_SIZE_CAP:\n                rows.append(_row(base, m, passes,\n                                 **metrics.blank(metrics.STATUS_SKIPPED,\n                                                 f\"n={n} > cap {config.DIRECT_SIZE_CAP}\")))\n                continue\n            if m.dense and A_dense is None:\n                rows.append(_row(base, m, passes,\n                                 **metrics.blank(metrics.STATUS_SKIPPED, \"dense form not built\")))\n                continue\n            try:\n                t0 = time.perf_counter()\n                if passes:\n                    x, iters, conv, work = refine(m.fn, operand, b, passes=passes)\n                else:\n                    x, iters, conv, work = m.fn(operand, b)\n                elapsed = time.perf_counter() - t0\n                scored = metrics.score(A, x, b, x_true, b_norm, xt_norm, conv)\n                rows.append(_row(base, m, passes, runtime_sec=elapsed, iterations=iters,\n                                 **work.as_dict(), **scored))\n            except SolverNotApplicable as e:\n                rows.append(_row(base, m, passes,\n                                 **metrics.blank(metrics.STATUS_NOT_APPLICABLE, str(e))))\n            except (MemoryError, RuntimeError, ValueError, ZeroDivisionError) as e:\n                rows.append(_row(base, m, passes,\n                                 **metrics.blank(metrics.STATUS_ERROR, f\"{type(e).__name__}: {e}\")))\n        if verbose:\n            last = rows[-1]\n            print(f\"      {m.name:<24s} {last['status']}\")\n\n    spec = {**base, \"status\": \"analysed\"}\n    if with_spectral:\n        try:\n            rho_j, how_j = spectral.spectral_radius(A, \"jacobi\")\n            rho_g, how_g = spectral.spectral_radius(A, \"gauss_seidel\")\n            spec.update(spectral.classify(A, rho_j, rho_g),\n                        rho_jacobi_method=how_j, rho_gs_method=how_g)\n        except (MemoryError, ValueError) as e:\n            spec.update(status=f\"spectral_failed: {type(e).__name__}\")\n\n    return rows, spec\n\n\ndef run_full_benchmark(dataset_root, results_csv, spectral_csv=None,\n                       refinement_passes=(0, 1), with_spectral=True,\n                       seed=None, verbose=True):\n    \"\"\"Sweep the whole corpus, checkpointing after every matrix.\"\"\"\n    entries = discover_matrices(dataset_root)\n    all_rows, all_spec = [], []\n    t0 = time.perf_counter()\n\n    for i, entry in enumerate(entries, 1):\n        if verbose:\n            print(f\"\\n[{i}/{len(entries)}] {entry['domain']}/{entry['name']}\"\n                  f\"   ({(time.perf_counter() - t0) / 60:.1f} min elapsed)\")\n        rows, spec = run_one_matrix(entry[\"domain\"], entry[\"name\"], entry[\"path\"],\n                                    refinement_passes=refinement_passes,\n                                    with_spectral=with_spectral, seed=seed, verbose=verbose)\n        all_rows.extend(rows)\n        all_spec.append(spec)\n        pd.DataFrame(all_rows).to_csv(results_csv, index=False)     # checkpoint\n        if spectral_csv:\n            pd.DataFrame(all_spec).to_csv(spectral_csv, index=False)\n\n    results = pd.DataFrame(all_rows)\n    if verbose:\n        expected = len(entries) * len(METHODS) * len(refinement_passes)\n        print(f\"\\nSweep complete in {(time.perf_counter() - t0) / 60:.1f} min\")\n        print(f\"  rows {len(results):,} of {expected:,} expected\"\n              f\"  |  matrices {results['matrix'].nunique()} of {len(entries)}\")\n        print(results[\"status\"].value_counts().to_string())\n    return results, pd.DataFrame(all_spec)\n\n\ndef summarise(results, passes=0):\n    \"\"\"Applicability and conditional success per method, as separate columns.\n\n    These are never multiplied into a single rate: doing that is what reported\n    Conjugate Gradient at 10.4% when its conditional success is 83.9%, and\n    Cholesky at 10.1% when it solves every system it applies to.\n    \"\"\"\n    subset = results[results[\"refinement_passes\"] == passes]\n    return pd.DataFrame([metrics.rates(subset, m) for m in METHOD_NAMES\n                         if m in set(subset[\"method\"])])\n", "__init__": "\"\"\"SolveBench: direct and iterative linear solvers benchmarked on real sparse matrices.\n\nThis package is the single source of truth for what the benchmark runs. The\nKaggle notebook is generated from it by ``tools/build_notebook.py`` rather than\ncarrying its own copy of the solvers, which is how the notebook and this library\npreviously drifted apart -- different size caps, different solver sets, and a\ndataset layout the library could not even find.\n\nSolver contract: every method returns ``(x, iterations, reported_converged, work)``.\nThe solver's own convergence claim is recorded but never decides the outcome;\n:func:`solvebench.metrics.score` does that, from the residual it measures itself.\n\"\"\"\nfrom . import (benchmark, config, direct_solvers, io_utils, iterative_solvers,\n               metrics, reference_solvers, refinement, spectral)\nfrom .benchmark import METHODS, METHOD_NAMES, run_full_benchmark, run_one_matrix, summarise\nfrom .direct_solvers import SolverNotApplicable\nfrom .io_utils import discover_matrices, load_matrix, make_ground_truth\nfrom .metrics import score\nfrom .refinement import refine\n\n__all__ = [\n    \"benchmark\", \"config\", \"direct_solvers\", \"io_utils\", \"iterative_solvers\",\n    \"metrics\", \"reference_solvers\", \"refinement\", \"spectral\",\n    \"METHODS\", \"METHOD_NAMES\", \"run_full_benchmark\", \"run_one_matrix\", \"summarise\",\n    \"SolverNotApplicable\", \"discover_matrices\", \"load_matrix\", \"make_ground_truth\",\n    \"score\", \"refine\",\n]\n"}""")

_pkg = pathlib.Path("/kaggle/working/solvebench")
if not pathlib.Path("/kaggle").exists():
    _pkg = pathlib.Path("solvebench")
_pkg.mkdir(parents=True, exist_ok=True)
for _name, _src in _SOURCES.items():
    (_pkg / f"{_name}.py").write_text(_src, encoding="utf-8")
sys.path.insert(0, str(_pkg.parent))

import solvebench
from solvebench import benchmark, config, io_utils, metrics, reordering, spectral
print(f"library loaded: {len(_SOURCES)} modules, "
      f"{len(benchmark.METHODS)} methods")
for _m in benchmark.METHODS:
    print(f"   {_m.family:<20s} {_m.name}")

In [ ]:
import platform, numpy, scipy, pandas, multiprocessing, json, time
from datetime import datetime

ON_KAGGLE = pathlib.Path("/kaggle").exists()
ENV = {
    "started": datetime.now().isoformat(),
    "on_kaggle": ON_KAGGLE,
    "python": platform.python_version(),
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "pandas": pandas.__version__,
    "cpu_count": multiprocessing.cpu_count(),
    "platform": platform.platform(),
}
for k, v in ENV.items():
    print(f"  {k:<12s} {v}")

# scipy version is recorded deliberately. Kaggle runs 1.16.3 where local setups
# often run 1.17.x, and the difference is not cosmetic: spilu on a structurally
# singular matrix raises a catchable error on 1.17 but exhausts memory and gets
# the process OS-killed on 1.16.3. Two full sweeps died that way.

In [ ]:
DATA_ROOT = None
for _candidate in (pathlib.Path("/kaggle/input"), pathlib.Path("dataset_large")):
    if not _candidate.exists():
        continue
    _hits = list(_candidate.rglob("*.mtx"))
    if _hits:
        # Walk up from any matrix to the directory holding the domain folders.
        DATA_ROOT = _hits[0].parent.parent
        break

if DATA_ROOT is None:
    # A dataset attached seconds ago may not have propagated yet; that race
    # produced an empty /kaggle/input and a FileNotFoundError on an early run.
    print("NO MATRICES FOUND. Contents of /kaggle/input:")
    for _p in sorted(pathlib.Path("/kaggle/input").rglob("*"))[:40]:
        print("   ", _p)
    raise FileNotFoundError("dataset not attached or not yet propagated")

MATRICES = io_utils.discover_matrices(DATA_ROOT)
print(f"corpus root : {DATA_ROOT}")
print(f"matrices    : {len(MATRICES)}")
print(f"domains     : {len({m['domain'] for m in MATRICES})}")

# Off Kaggle the notebooks are only ever executed by tools/test_notebooks.py, so
# their output is scratch and must not land beside the real results. It used to write
# into a directory that was also make_figures' default input, which meant running the
# safety check silently overwrote the tables the figures were built from.
OUT_DIR = pathlib.Path("/kaggle/working/output" if ON_KAGGLE else ".nbscratch/output")
(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
print(f"output      : {OUT_DIR}")

In [ ]:
T0 = time.perf_counter()
import numpy as np
import pandas as pd

# The question this notebook exists to answer: how many systems does *choosing* the
# diagonal move from "no stationary method works" to "solved"?
#
#   none  -- the matrix as it arrives, the baseline every earlier result uses
#   mc64  -- the established permutation, maximising the product of |diagonal| entries
#   best  -- compute mc64, min-sum and bottleneck, keep whichever gives the smallest
#            estimated rho(T_GS). "none" is among the candidates, so this can never do
#            worse than leaving the matrix alone.
CONDITIONS = ("none", "mc64", "best")
STATIONARY = {"Jacobi": solvebench.iterative_solvers.jacobi,
              "Gauss-Seidel": solvebench.iterative_solvers.gauss_seidel,
              "SOR": solvebench.iterative_solvers.sor}

# Set by the generator. The probe variant trades corpus coverage and the exact spectra
# for a run short enough to confirm the pipeline works before the full sweep finishes.
PROBE_N = 200
WITH_EXACT_SPECTRA = False

_SHORT = {"solved": "ok", "not_applicable": "n/a", "diverged": "div",
          "inaccurate": "bad", "error": "err"}


def _outcome(r):
    """One compact token per method, e.g. ``Jac:ok(37)`` or ``SOR:cap(10000)``.

    ``cap`` and ``stop`` are both did_not_converge but mean opposite things: cap ran the
    full iteration budget and was still going, stop was abandoned early by the
    divergence guard. Collapsing them hides which one happened.
    """
    name = str(r.get("method", "?"))[:3]
    it = r.get("iterations")
    shown = int(it) if isinstance(it, (int, float)) and it == it else "-"
    status = str(r.get("status"))
    if status == "did_not_converge":
        tag = "cap" if shown != "-" and shown >= config.MAX_ITERATIONS else "stop"
    else:
        tag = _SHORT.get(status, status[:3])
    return "{}:{}({})".format(name, tag, shown)


rows = []
csv = OUT_DIR / "tables" / "reordering_study.csv"
order = sorted(MATRICES, key=lambda e: e["path"].stat().st_size)

if PROBE_N and PROBE_N < len(order):
    # Stratified by size decile so the sample keeps the corpus's spread rather than
    # filling up with small, fast matrices.
    rng = np.random.default_rng(20260910)
    edges = np.linspace(0, len(order), 11).astype(int)
    picked = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        bucket = order[lo:hi]
        take = min(PROBE_N // 10, len(bucket))
        picked += [bucket[j] for j in rng.choice(len(bucket), size=take, replace=False)]
    order = sorted(picked, key=lambda e: e["path"].stat().st_size)
    print(f"probe: {len(order)} matrices sampled across 10 size deciles (seed 20260910)")

# Kaggle kills a session at 12 hours without warning. Stop cleanly before that, write
# the results, and say so -- a run that reports "budget reached at matrix 812" is worth
# far more than one that simply vanishes. The previous attempt was cancelled at the
# limit and only survived because Kaggle happened to publish the checkpoint.
TIME_BUDGET_H = 11.0
_total_nnz = sum(e["path"].stat().st_size for e in order)

print("=" * 78)
print("REORDERING STUDY -- convergence-oriented diagonal selection")
print("=" * 78)
print(f"  matrices          : {len(order)}")
print(f"  conditions        : {', '.join(CONDITIONS)}")
print(f"  methods           : {', '.join(STATIONARY)}")
print(f"  iteration cap     : {config.MAX_ITERATIONS:,}   tolerance {config.TOLERANCE:g}")
print(f"  exact spectra     : {'yes, n <= %d' % config.SPECTRAL_EXACT_CAP if WITH_EXACT_SPECTRA else 'no'}")
print(f"  wall-clock budget : {TIME_BUDGET_H} h  (Kaggle cancels at 12 h)")
print(f"  expected runtime  : 3.5-4.5 h typical, 9 h worst case")
print(f"  checkpoint        : {csv.name} rewritten after every matrix")
print("=" * 78, flush=True)

for i, entry in enumerate(order, 1):
    base = {"domain": entry["domain"], "matrix": entry["name"]}
    try:
        A = io_utils.load_matrix(entry["path"])
    except Exception as e:
        rows.append({**base, "status": f"load_failed: {type(e).__name__}"})
        continue
    n = A.shape[0]
    base.update(n=n, nnz=A.nnz)
    zr, zc = io_utils.structural_singularity(A)
    if zr or zc:
        rows.append({**base, "status": "structurally_singular"})
        pd.DataFrame(rows).to_csv(csv, index=False)
        continue

    x_true, b = io_utils.make_ground_truth(A)
    bn = np.linalg.norm(b) or 1.0
    xn = np.linalg.norm(x_true) or 1.0

    # Progress is printed BEFORE the work, and again after each condition. The previous
    # run printed one line per matrix only once all three conditions were done, so when
    # it stalled inside a permutation the log simply stopped -- twelve hours of silence
    # and no way to tell which matrix or which step was responsible. Announcing the
    # matrix first means the last line in the log always names whatever is stuck.
    elapsed = (time.perf_counter() - T0) / 60
    rate = i / max(elapsed, 1e-9)
    eta = (len(order) - i) / rate if rate > 0 else float("nan")
    print(f"[{i:>4}/{len(order)}] {100.0 * i / len(order):5.1f}%  {entry['name']:<24} "
          f"n={n:<6} nnz={A.nnz:<9} | {elapsed / 60:5.2f}h elapsed"
          f" | ETA {eta / 60:5.2f}h | projected total {(elapsed + eta) / 60:5.2f}h",
          flush=True)

    for cond in CONDITIONS:
        print(f"    {cond:<6} ...", flush=True, end="")
        t0 = time.perf_counter()
        try:
            A2, b2, info = reordering.select_diagonal(A, b, objective=cond)
        except Exception as e:
            rows.append({**base, "condition": cond,
                         "status": f"reorder_failed: {type(e).__name__}"})
            continue
        setup = time.perf_counter() - t0
        zeros = int((np.abs(A2.diagonal()) < 1e-14).sum())

        # Exact spectra only for the two conditions the write-up compares, and only
        # where the dense route is affordable. Selection itself uses a cheap estimate.
        rho_j = rho_g = np.nan
        if (WITH_EXACT_SPECTRA and cond in ("none", "best")
                and n <= config.SPECTRAL_EXACT_CAP and zeros == 0):
            try:
                rho_j, _ = spectral.spectral_radius(A2, "jacobi")
                rho_g, _ = spectral.spectral_radius(A2, "gauss_seidel")
            except Exception:
                pass

        for mname, fn in STATIONARY.items():
            # worst_ratio_* is the quantity the method is actually optimising, and the
            # one the convergence guarantee is stated in: below 1 the permuted matrix is
            # strictly diagonally dominant, so Jacobi AND Gauss-Seidel are guaranteed to
            # converge. Recording it for every matrix is what turns "the permutation
            # helped" into "the permutation bought a guarantee, on this many matrices".
            row = {**base, "condition": cond, "method": mname,
                   "chosen": info.get("chosen", cond), "setup_sec": setup,
                   "zero_diagonal": zeros, "rho_jacobi": rho_j, "rho_gauss_seidel": rho_g,
                   "permuted": info.get("permuted", False),
                   "perm_cost": info.get("cost", float("nan")),
                   "worst_ratio_before": info.get("worst_ratio_before", float("nan")),
                   "worst_ratio_after": info.get("worst_ratio_after", float("nan")),
                   **{f"ratio_{k}": v
                      for k, v in info.get("ratios", {}).items()}}
            if zeros:
                row.update(metrics.blank(metrics.STATUS_NOT_APPLICABLE, "zero diagonal"))
                rows.append(row)
                continue
            try:
                t1 = time.perf_counter()
                x, its, conv, work = fn(A2, b2)
                row.update(runtime_sec=time.perf_counter() - t1, iterations=its,
                           **work.as_dict(),
                           **metrics.score(A2, x, b2, x_true, bn, xn, conv))
            except solvebench.SolverNotApplicable as e:
                row.update(metrics.blank(metrics.STATUS_NOT_APPLICABLE, str(e)))
            except (MemoryError, RuntimeError, ValueError, ZeroDivisionError) as e:
                row.update(metrics.blank(metrics.STATUS_ERROR, f"{type(e).__name__}: {e}"))
            rows.append(row)

        made = rows[-len(STATIONARY):]
        solved = sum(1 for r in made if r.get("status") == metrics.STATUS_SOLVED)
        detail = " ".join(_outcome(r) for r in made)
        wr = info.get("worst_ratio_after", info.get("worst_ratio_before", float("nan")))
        print(f" {solved}/{len(STATIONARY)} solved | {detail} | perm {setup:6.2f}s"
              f" | total {time.perf_counter() - t0:7.2f}s"
              f" | ratio {wr:.3g}" + (f" | chose {info.get('chosen', cond)}"
                                      if cond == "best" else ""), flush=True)

    pd.DataFrame(rows).to_csv(csv, index=False)

    # A standing scoreboard every 25 matrices: the log should answer "is it working?"
    # without waiting for the end, and "will it finish?" without doing arithmetic.
    if i % 25 == 0 or i == len(order):
        d = pd.DataFrame(rows)
        if "condition" in d and "status" in d:
            tally = {c: int((d[(d.condition == c)].status == metrics.STATUS_SOLVED).sum())
                     for c in CONDITIONS if (d.condition == c).any()}
            spent = (time.perf_counter() - T0) / 3600
            print(f"    ---- after {i}/{len(order)}: solved "
                  + ", ".join(f"{c}={v}" for c, v in tally.items())
                  + f" | {spent:.2f}h spent, {spent / max(i, 1) * len(order):.2f}h projected"
                  + f" | {len(d):,} rows", flush=True)

    # Kaggle cancels at 12 h with no warning. Stop first, and say where we stopped.
    if (time.perf_counter() - T0) / 3600 > TIME_BUDGET_H:
        print(f"\n*** WALL-CLOCK BUDGET {TIME_BUDGET_H} h REACHED at matrix {i}"
              f"/{len(order)} ({entry['name']}). Stopping cleanly; "
              f"{len(rows):,} rows written to {csv.name}. ***", flush=True)
        break

study = pd.DataFrame(rows)
print(f"\nrows {len(study):,}")

In [ ]:
ok = study[study.method.notna()]
piv = (ok.assign(solved=ok.status == metrics.STATUS_SOLVED)
         .pivot_table(index=["matrix", "method"], columns="condition",
                      values="solved", aggfunc="max"))

print("=== systems solved, by condition ===")
for c in CONDITIONS:
    if c in piv:
        print(f"  {c:<6} {int(piv[c].sum()):>5}")

if "none" in piv and "best" in piv:
    gained = piv[(piv["none"] == 0) & (piv["best"] == 1)]
    lost = piv[(piv["none"] == 1) & (piv["best"] == 0)]
    print(f"\n  RESCUED (failed as given, solved after choosing the diagonal): {len(gained)}")
    print(f"  LOST    (solved as given, failed after)                      : {len(lost)}")
    if "mc64" in piv:
        mc = piv[(piv["none"] == 0) & (piv["mc64"] == 1)]
        print(f"  MC64 rescues                                                 : {len(mc)}")
        print(f"  ours rescues where MC64 does not                             : "
              f"{len(set(gained.index) - set(mc.index))}")
    print("\n  by method:")
    for m in STATIONARY:
        lvl = piv.index.get_level_values("method")
        if m not in set(lvl):
            continue
        sub = piv.xs(m, level="method")
        g = ((sub["none"] == 0) & (sub["best"] == 1)).sum()
        print(f"    {m:<14} {int(sub['none'].sum()):>4} -> {int(sub['best'].sum()):>4}"
              f"   (+{int(g)} rescued)")

print("\n=== which objective the selector picked ===")
print(ok[ok.condition == "best"].drop_duplicates("matrix").chosen.value_counts().to_string())

sp_rows = ok[(ok.condition.isin(["none", "best"])) & ok.rho_gauss_seidel.notna()]
if len(sp_rows):
    w = sp_rows.drop_duplicates(["matrix", "condition"]).pivot(
        index="matrix", columns="condition", values="rho_gauss_seidel").dropna()
    if {"none", "best"} <= set(w.columns):
        crossed = ((w["none"] >= 1) & (w["best"] < 1)).sum()
        print(f"\n=== rho(T_GS) crossing below 1 (n <= {config.SPECTRAL_EXACT_CAP}) ===")
        print(f"  matrices measured both ways  : {len(w)}")
        print(f"  rho >= 1 as given, < 1 after : {int(crossed)}")
        print(f"  median change in rho         : {(w['best'] - w['none']).median():.4g}")

one = ok[ok.condition == "best"].drop_duplicates("matrix")
have = [c for c in ("ratio_none", "ratio_mc64", "ratio_minsum", "ratio_bottleneck")
        if c in one.columns]
if have:
    print("\n=== strict diagonal dominance, bought by permutation ===")
    print("A worst row ratio below 1 IS strict diagonal dominance, which guarantees both")
    print("Jacobi and Gauss-Seidel converge. The bottleneck objective minimises exactly")
    print("that ratio, so if any row permutation makes the matrix dominant, it finds one:")
    print("the guarantee is reachable precisely when ratio_bottleneck < 1. MC64 maximises")
    print("the product of |diagonal| instead and carries no such statement.")
    for col in have:
        v = pd.to_numeric(one[col], errors="coerce")
        fin = v.replace([np.inf, -np.inf], np.nan)
        print(f"  {col[6:]:<11} dominant {int((v < 1).sum()):>4} / {int(v.notna().sum()):>4}"
              f"   median ratio {fin.median():.4g}"
              f"   undefined {int(np.isinf(v).sum()):>4}")
    if {"ratio_none", "ratio_bottleneck"} <= set(one.columns):
        a = pd.to_numeric(one.ratio_none, errors="coerce")
        c = pd.to_numeric(one.ratio_bottleneck, errors="coerce")
        print(f"\n  not dominant as given, dominant after bottleneck : "
              f"{int(((a >= 1) & (c < 1)).sum())}")
        print(f"  dominant as given, lost by bottleneck            : "
              f"{int(((a < 1) & (c >= 1)).sum())}")

piv.to_csv(OUT_DIR / "tables" / "reordering_pivot.csv")

In [ ]:
ENV["finished"] = datetime.now().isoformat()
ENV["runtime_minutes"] = (time.perf_counter() - T0) / 60
ENV["notebook"] = "solvebench-reordering-probe"
ENV["config"] = {k: getattr(config, k) for k in dir(config)
                 if k.isupper() and not k.startswith("_")}
(OUT_DIR / "logs" / "run_metadata.json").write_text(json.dumps(ENV, indent=2, default=str))

print("=" * 70)
print(f"solvebench-reordering-probe COMPLETE in {ENV['runtime_minutes']:.1f} min")
print("=" * 70)
for _t in ["reordering_study.csv", "reordering_pivot.csv"]:
    _f = OUT_DIR / "tables" / _t
    print(f"  {_t:<34s} {_f.stat().st_size/1024:>9.1f} KB" if _f.exists()
          else f"  {_t:<34s} MISSING")